In [ ]:
# --- Package imports ---
%load_ext autoreload
%autoreload 2

import os
os.environ.setdefault("LVMCORE_DIR", "/Users/droryn/prog/lvm/lvmcore")

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

# Notebook cwd is skysub/, so mlp_predictor/ and sky_decomp/ are top-level packages.
from mlp_predictor import config, data, wavelengths, compressor, trainer, diagnostics
from mlp_predictor.config import DataConfig, PipelineConfig

# Journal-style plotly axes (as in the pre-refactor notebook).
for _ax in (pio.templates["plotly_white"].layout.xaxis,
            pio.templates["plotly_white"].layout.yaxis):
    _ax.showgrid = False
    _ax.showline = True
    _ax.mirror = True
    _ax.linecolor = "black"
    _ax.linewidth = 1
    _ax.ticks = "inside"
    _ax.zeroline = False
pio.templates.default = "plotly_white"

FACTOR = 1e14

# Publication figure output: every figure in this notebook is shown through
# show_with_pdf_export() below, which saves here as a vector PDF.
FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)


def show_with_pdf_export(fig, path):
    """Display `fig`, then immediately save it to `path` as a vector PDF, at
    the figure's OWN print size.

    No buttons (2026-09-28, simplified from the earlier Preview/Export
    pair). Those existed only so a figure could be interactively adjusted --
    panned, zoomed -- before committing to an export of that exact view. That
    stopped being possible earlier the same day, when go.FigureWidget was
    dropped for a plain go.Figure (Plotly 6.x's FigureWidget is an
    `anywidget.AnyWidget` subclass, and VS Code's Jupyter ipywidgets renderer
    does not support anywidget's widget-loading protocol, so every cell in
    this notebook failed to display on a fresh kernel): a plain Figure's
    pan/zoom lives entirely in the browser/webview with no comm channel back
    to the kernel, so a button click could no longer capture anything but the
    figure's own pre-built ranges anyway -- which is exactly what saving
    immediately also gives, without the now-pointless click.
    """
    _export_w, _export_h = fig.layout.width, fig.layout.height
    # Spell out tick numbers (4000, not plotly's default SI "4k") on every x
    # axis -- the wavelength axes in particular.
    fig.update_xaxes(exponentformat='none')
    fig.show()
    # Render twice, keep the second: kaleido's MathJax typesetting is
    # occasionally still mid-render when a complex figure (many $...$ spans
    # -- title/legend/axis text across several panels) gets snapshotted,
    # silently leaving raw "$\text{...}$" source in the output instead of the
    # rendered glyphs. Not a code bug in this figure -- reproduced on a plain
    # go.Figure with no widget involved, and it is intermittent (same figure,
    # same call, sometimes clean) -- so retrying is the only defence
    # available; a warm second pass has not been observed to fail.
    fig.write_image(str(path), width=_export_w, height=_export_h)
    fig.write_image(str(path), width=_export_w, height=_export_h)
    print(f'Saved {path}')
    return fig

# --- Corpus + artifact locations: the one place to switch corpora -----------
# Every input path -- meta, per-arm coefficients, per-arm decomposition, the
# basis grid and the coefficient wavelengths -- is derived from DECOMP_DATA_ROOT and
# DECOMP_STEM by mlp_predictor.config.DataConfig, and the trained ensemble is
# written beside the corpus it was trained on, so a single edit here moves the
# whole pipeline.
#
#   root          stem                                     rows   selection
#   'new-oh-3'    lvmsframe_median_stack_1.2.1_p40_p70    17 260  faint-fibre percentile
#   'gaia-stars'  lvmsframe_median_stack_1.2.1_gaia1over100 14 469  GAIA stellar SB < 1% of sky
#
# BOTH must be set when switching: the stem differs between corpora, and a root
# change alone silently looks for the wrong filenames under the new root.
#
# The coefficient wavelengths are no longer cached: they are read off the
# decomposer design matrix in ~1 s per run, from THIS root's own basis, so they
# cannot go stale.  Compressors and the ensemble are still never transferable
# between corpora -- the moon/zodi split and its sum move whenever the
# decomposition changes.
# --- DECOMPOSITION VARIANT -------------------------------------------------
# Two decomposition flavours of the same input stack, switchable here:
#
#   'split_zodi'  the production LSF-surface-iterative split-zodi fit
#                 (gaia-stars-mask-cont2, W = 0.15 diffuse/OH cap)
#   'telluric'    palace-aijc-vnf-split-zodi-lsf-spline2d: PALACE VNF lines
#                 with OH grouped by (v_upper, N_upper, F_upper), a continuous
#                 M-spline 2-D LSF, and TELLURIC ABSORPTION included in the fit
#
# The coefficient vector is the SAME 388 names in the same order, so the ML side
# is drop-in and every coefficient-space number is directly comparable between
# the two. What differs is the RECONSTRUCTION path, in two ways that both bite:
#
#   * the LSF is stored as a continuous M-spline density rather than a discrete
#     11-tap surface, so the iterative class cannot read it at all; and
#   * every family matrix is divided by that ROW's DRP transmission, so the
#     design matrix is per-row and cannot be hoisted (~0.17 s/row/arm to
#     rebuild, about 4 min for a 500-row sample across three arms).
#
# `mlp_predictor.data.DECOMP_VARIANTS` holds the suffixes;
# `make_telluric_row_lookup` supplies the per-row transmission, and the
# diagnostics pick it up from the `TELLURIC_ROW_FOR` extra installed below.
#
# Integrated amplitudes A_g = c_g . v_g keep the TELLURIC-FREE template
# integrals in both flavours: the coefficient multiplies the same physical
# template either way and the transmission is a per-row correction applied to
# it, so the plain integrals are both the physical amplitude and the only
# choice that leaves the two corpora comparable.
# 2026-09-18: the non-telluric variant was removed from this branch; the
# telluric fit is the only supported decomposition (see data.DECOMP_VARIANTS).
# 2026-09-24: 'telluric-palacecorr' is the same telluric fit with the SkyFar
# ridge-corrected OH line strengths (decompose_parallel's default from that
# date).  Same 388 coefficients; the variant entry selects the matching OH line
# file for every reconstruction, which is what makes its flux numbers valid.
DECOMP_VARIANT = 'telluric-palacecorr'

_VARIANT_ROOT = {
    'telluric':            'gaia-stars-mask-telluric-chi2-1.3.2',
    'telluric-palacecorr': 'gaia-stars-mask-telluric-chi2-palacecorr',
}
if DECOMP_VARIANT not in _VARIANT_ROOT:
    raise ValueError(f"DECOMP_VARIANT must be one of {sorted(_VARIANT_ROOT)}; "
                     f"got {DECOMP_VARIANT!r}")
DECOMP_DATA_ROOT = _VARIANT_ROOT[DECOMP_VARIANT]
DECOMP_STEM = 'lvmsframe_median_stack_1.3.2_gaia1over100'
# Same filename under both roots -- the root already separates them, so the two
# variants cannot overwrite each other's ensemble.
ENSEMBLE_FILENAME = 'mlp_ensemble_split_zodi_cont-1.3.2.pt'   # 'new-oh' legacy name: mlp_ensemble_split_zodi_new_oh.pt

# --- Physical-model moon transfer ratio as a context feature ---------------
# Adds ONE ctx column, log10(moon_model_sci / moon_model_near), gated to 0 on
# moon-down rows, from the cache built by mlp_predictor.moon_model_cache.
# A/B on 2 x 10 seeds, identical rows/split/seeds: moon amplitude MAD on
# MOON-UP test rows 0.01492 -> 0.00771 (-48.4%), p90 -23.6%, zodi MAD -16.8%,
# continuum +5.0%, mean_eRMSE 7.858 -> 7.774.  The moon-up MAD of 0.0077 beats
# the physical model's own 0.0119, i.e. the net does better than either the
# model or its old inputs alone.
# The cache is per corpus and BUILDS ITSELF on first use -- expect a one-off
# ~26 min on 8 cores the first time a new corpus is
# trained on, and ~3 min again when the first every10 diagnostic cell runs
# (the corpus and its every10 subsample need separate caches, since row N of
# one is not row N of the other).  Build them ahead of time with
#   python -m mlp_predictor.moon_model_cache <prefix> --n-workers 8
# A cache that exists but does NOT match the corpus is never silently rebuilt;
# it raises and names the offending row, because a mismatch is as likely to be
# the wrong prefix as a stale file.
# Set False to A/B.  NOTE this changes ctx_names, so it invalidates any saved
# ensemble -- cell 10's guard will refuse to load one.
USE_MOON_MODEL_FEATURE = True

cfg = PipelineConfig()
cfg.data.decomp_data_root = DECOMP_DATA_ROOT
cfg.data.decomp_stem = DECOMP_STEM
# The suffix is NOT a DataConfig default any more -- it identifies the variant.
cfg.data.decomp_suffix = data.DECOMP_VARIANTS[DECOMP_VARIANT]['suffix']
USE_TELLURIC_RECON = bool(data.DECOMP_VARIANTS[DECOMP_VARIANT]['telluric'])
SAVED_ENSEMBLE_PATH = f"{DECOMP_DATA_ROOT}/{ENSEMBLE_FILENAME}"

print(f"Corpus:   {cfg.data.decomp_data_root}/{cfg.data.decomp_stem} "
      f"(suffix={cfg.data.decomp_suffix!r})")
print(f"Ensemble: {SAVED_ENSEMBLE_PATH}")





In [ ]:
# --- Load triplet + apply filters + augment context ---
triplet = data.build_triplet_coef_dataset(
    input_fits_path=cfg.data.input_fits_meta,
    sky_near_decomp_fits_path=cfg.data.coef_fits("sky1"),
    sky_far_decomp_fits_path=cfg.data.coef_fits("sky2"),
    sci_decomp_fits_path=cfg.data.coef_fits("sci"),
    context_columns=list(cfg.data.context_columns),
    return_chi2=True,
)
print("Loaded triplet shapes:",
      {k: v.shape for k, v in triplet.items()
       if hasattr(v, "shape") and k in ("coef_near","coef_far","coef_sci","ctx_sci")})

# Native wavelength grid, for the moon/zodi role-reversal check below.
from astropy.io import fits as _fits_w
with _fits_w.open(cfg.data.input_fits_for_basis) as _hw:
    _wave_native = np.asarray(_hw["WAVE"].data, dtype=float)
    _wave_native = _wave_native if _wave_native.ndim == 1 else _wave_native[0]

filtered_triplet = data.apply_triplet_filters(
    triplet,
    thin_every_n=1, chi2_qmax=90.0, chi2_min=0.0, chi2_max=10.0,
    hard_coef_bounds={"feo": (0.0, 36.01), "atom_k": (0.0, 10.01)},
    # 2026-09-11: kappa 6 -> 8, oh_kappa 4 -> 6.  Both gates were cutting valid
    # extreme data, not broken decompositions.  Discriminator was the
    # DECOMPOSITION-S OWN fit (absolute single-fibre photon chi2 from
    # BESTFIT_LSF vs FLUX_SCI) plus the median FRACTIONAL residual -- chi2
    # alone is confounded, since at fixed fractional accuracy photon chi2
    # grows with brightness.  On every10, of the rows passing every OTHER
    # gate: the other gates find real failures (chi2 96.1 vs 3.65 kept, 26x,
    # p90 3610) while these two do not.
    #
    #   cut by kappa-sigma (n=37): chi2 1.26x kept, fractional residual
    #     1.02x -- indistinguishable.  59% sit at worst |z| < 8 against a
    #     kappa=6 threshold, and the offenders are ATOM_Orc_OI0845 / OI0777 /
    #     ATOM_Og / ATOM_K, i.e. bright atomic airglow lines.  The gate is
    #     MIS-SPECIFIED: it uses a non-robust mean/std and median std/MAD over
    #     its 30 columns is 13.46 against the Gaussian 1.483.  The heavy tails
    #     are in the MOON knots (std/MAD 29-104, skew ~2 -- bimodal, ~0
    #     moon-down and large moon-up), which inflates their sigma and
    #     DISARMS the gate there; the atomic columns are the most Gaussian
    #     (std/MAD 2.14) and so carry the tightest effective threshold.  It
    #     had become an accidental atomic-line gate.  Same failure the code
    #     already documents for the OH block, still present in moon.
    #
    #   cut by the OH-MAD gate (n=34): airmass 1.63 (p86 of kept), alt 37.8
    #     deg (p14), van Rhijn 1.60 (p86) -- exactly where the OH column is
    #     physically largest.  The gate scalar sits 3.56x the kept median
    #     against a threshold at 3.10x, i.e. 1.15x PAST a threshold built to
    #     catch runaways at 1e6-1e15x.  Their fits are 1.45x worse
    #     fractionally, plausibly because LSF and telluric treatment degrade
    #     at high airmass: real data the CURRENT decomposition fits less
    #     well, not bad data.
    #
    # Extremity is coherent across all three arms of the same exposure on 84%
    # of cut rows vs 62% of kept -- a QP failure would hit one arm, not three.
    #
    # Recovers ~54 of the 73 cut every10 rows, concentrated in atomic-line
    # brightness and in OH at high airmass -- and OH carries 72-78% of the
    # band-integrated flux error tail.  NOT yet shown to improve the model:
    # the OH-MAD rows also bring 1.45x noisier targets, and the LMC/SMC cut
    # is precedent that a real explanation can still lose a controlled A/B.
    # Revert to kappa=6.0 / oh_kappa=4.0 if this arm loses.
    kappa=8.0, kappa_iter=3, oh_kappa=6.0, oh_kappa_iter=3,
    exclude_field_regions=[data.LMC_EXCLUSION, data.SMC_EXCLUSION],
    # Drop rows where the moon and zodi splines swapped roles in ANY arm: the
    # moon coefficients of such a row describe zodiacal light, so they teach
    # the network the wrong geometry mapping rather than merely adding noise.
    # Tested only where both families carry flux (moon share in [0.05, 0.95]),
    # so dark-time rows -- pinned near a 0.02 moon share by the decomposition
    # priors -- are exempt by construction.
    reversal_decomp_fits={
        "near": cfg.data.decomp_fits("sky1"),
        "far":  cfg.data.decomp_fits("sky2"),
        "sci":  cfg.data.decomp_fits("sci"),
    },
    reversal_wave=_wave_native,
    reversal_min_component_frac=0.05,
    reversal_min_separation=0.0,
    # Drop rows whose SCIENCE fibre carries continuum the sky basis cannot
    # represent.  The Moon_bs spline is the only flexible continuum in the
    # basis -- the zodi total is pinned by the Leinert anchor and the three
    # diffuse species have fixed shapes -- so field continuum lands in the
    # moon coefficients and that row's moon target stops describing scattered
    # moonlight.  Measured on this corpus: rho = +0.405 between the colour
    # excess and the sci-minus-near moon spline colour distortion, against
    # +0.034 for the far-minus-near sky-arm control, and rho = -0.331 against
    # the signed ML moon amplitude error on moon-up rows.  Costs 7.1% of rows.
    # NOTE this needs the full stack, not input_fits_for_basis (every10) or
    # input_fits_meta (no flux at all).
    colour_excess_input_fits=cfg.data.input_fits_flux,
    colour_excess_max=data.SCI_COLOUR_EXCESS_MAX,
    # Drop rows where the QP collapsed the whole diffuse block (HO2+FeO+O2Ac)
    # to ~0 in any arm.  188 sci rows on this corpus (1.30%), genuinely zero
    # rather than faint, and overwhelmingly dark time (10% moon-up against 50%
    # corpus-wide) -- the zodi/diffuse degeneracy taken to its limit, with the
    # Zodi_bs spline carrying the whole continuum.  A target of ~2e-6 against a
    # near-arm 322 measures nothing; those rows alone put the continuum p95
    # log-error at 8 dex.  Existing gates already remove ~66% of them (their
    # median sci chi2 is 1.98 against 0.185), so the marginal cost is small.
    diffuse_zeroed_frac=data.DIFFUSE_ZEROED_FRAC,
)

data._augment_triplet_with_ecliptic(filtered_triplet, force=True, meta_fits_path=cfg.data.input_fits_meta)
data._augment_triplet_with_physics_priors(filtered_triplet, force=True)
# MOON-MODEL-CTX-V1: the frozen physical model's moon transfer ratio.  Last of
# the augments so the feature is always the final ctx column, which keeps the
# every10 diagnostic cells' layout identical to the training one.
if USE_MOON_MODEL_FEATURE:
    data._augment_triplet_with_moon_model(
        filtered_triplet, cfg.data.decomp_prefix, force=True)
print(f"Augmented ctx: n_ctx={len(filtered_triplet['ctx_names'])}")


In [ ]:
# Preview pre- vs post-filter coefficient histograms (no ML state needed).
from mlp_predictor import diagnostics  # local import: survives kernel-restart edge cases

_tmp = diagnostics.Diagnostics(diagnostics.DiagnosticsContext(
    filtered_triplet=filtered_triplet,
    extras={"triplet": triplet},
)).coef_hist_prepost()


In [ ]:
# --- Per-row joint covariance blocks (COEF_COV_MOON / COEF_COV_ZODI) ---
import numpy as _np
from astropy.io import fits as _fits

def _load_cov_for_row_index(decomp_path, row_index):
    row_index = _np.asarray(row_index, dtype=_np.int64)
    with _fits.open(decomp_path) as _h:
        _moon = (_np.asarray(_h["COEF_COV_MOON"].data[row_index], dtype=_np.float64)
                 if "COEF_COV_MOON" in _h else None)
        _zodi = (_np.asarray(_h["COEF_COV_ZODI"].data[row_index], dtype=_np.float64)
                 if "COEF_COV_ZODI" in _h else None)
    return _moon, _zodi

_row_idx_cov = _np.asarray(filtered_triplet["row_index"], dtype=_np.int64)
for _arm, _path in {
    "near": cfg.data.decomp_fits("sky1"),
    "far":  cfg.data.decomp_fits("sky2"),
    "sci":  cfg.data.decomp_fits("sci"),
}.items():
    _m, _z = _load_cov_for_row_index(_path, _row_idx_cov)
    filtered_triplet[f"coef_cov_moon_{_arm}"] = _m
    filtered_triplet[f"coef_cov_zodi_{_arm}"] = _z
print("Per-row joint covariance loaded (or None where COEF_COV_* HDU absent).")


In [ ]:
# --- Populate coefficient wavelengths + effective extinction ---
# `decomp_suffix` selects WHICH BASIS the wavelengths are built from.  Every
# variant uses the same 388 coefficient NAMES, so the name list alone cannot
# tell two bases apart -- but the telluric fit groups OH by (v_upper, N_upper,
# F_upper) and its per-stick centroids differ from the split-zodi ones by a
# median 46.9 A (p90 288 A, max 859 A, 310 of 357 sticks past 2 A).  Those
# wavelengths set the per-coefficient extinction and van Rhijn geometry, so
# building from the wrong basis is not cosmetic.
#
# There is no longer a wavelength cache on disk: the centroids and k_eff are
# read off the decomposer's design matrix in ~1 s, so the cache saved ~1 s per
# run while keeping alive the failure mode above (an .npz copied between
# corpora passed the name check and carried the wrong OH centroids).  Any
# leftover coef_wavelengths_basis_v4.npz in a corpus directory is now unused
# and can be deleted.
ext = wavelengths.resolve_wavelengths_and_extinction(
    filtered_triplet,
    input_fits_for_basis=cfg.data.input_fits_for_basis,
    use_fitted_extinction=True,
    verbose=True,
    decomp_suffix=cfg.data.decomp_suffix,
)
group_indices = ext.group_indices
N_MOON_KNOTS, SPLIT_ZODI, N_ZODI_KNOTS = wavelengths.infer_spline_knots(
    filtered_triplet["coef_names"])


In [ ]:
# --- Fit per-group compressors on a moon-phase-stratified split ---
from mlp_predictor.ml_utils import moon_phase_deg_from_ctx, split_indices_by_moon_phase

_moon_phase = moon_phase_deg_from_ctx(filtered_triplet)
_split_tr, _split_va, _split_te = split_indices_by_moon_phase(
    filtered_triplet["obstime_mjd"], _moon_phase, seed=42)

group_compressors, compress_geom_kwargs = compressor.fit_all_group_compressors(
    filtered_triplet, group_indices,
    train_idx=_split_tr, held_idx=_split_va,
    xarm_threshold=compressor.COMPRESSION_XARM_THRESHOLD,
    verbose=True,
)
filtered_triplet["compress_train_idx"] = _split_tr
filtered_triplet["compress_val_idx"] = _split_va
filtered_triplet["compress_test_idx"] = _split_te


In [ ]:
# --- Context vs decomposition-amplitude relationship grid -------------------
# Structure borrowed from mlp_predictor/diagnostics_cells/relationship_scatter_
# matrix.py (row/column layout, cyclic-context decoding via
# data._decode_cyclic_context), but rewritten for the FINAL post-augmentation
# context set and for a physical-decomposition question rather than an
# ML-performance one: how does each coefficient GROUP's flux amplitude relate
# to each context variable, on the sky-pointing decompositions themselves --
# no trained ensemble or prediction involved.
#
# Row selection: restricted to filtered_triplet['compress_train_idx'], the
# same row set Trainer.run_ensemble actually trains on -- not every row that
# survived apply_triplet_filters, since val/test rows are held out from
# learning (used for early stopping / the compressor fit / final scoring, not
# for fitting network weights). That index only exists once the compressor
# cell below has run, so THIS CELL MUST RUN AFTER IT, not right after
# wavelengths/group_indices as in an earlier version.
#
# Amplitude per group: the FLUX-INTEGRATED functional
# A_g = sum_k c_k * v_k, v_k = template_k(lambda).sum(),
# i.e. mlp_predictor/diagnostics_cells/amplitude_error_vs_ctx.py's own
# definition (2026-09-29: switched from a plain mean-of-coefficients proxy to
# match that diagnostic everywhere). v_k is looked up by COEFFICIENT NAME
# from the decomposer's own static per-family basis matrices
# (matrix_moon/zodi/diffuse/oh/atom/orc), never assumed to share row order
# with group_indices: the decomposer's own 'atom'/'orc' family split does NOT
# match this project's atomic/ionospheric ML grouping (ATOM_N and ATOM_Or are
# 'atom' to the decomposer, 'ionospheric' to the transfer), so every
# coefficient is realigned by name. O2_b01 has no static template (it is
# fit per row as VECTOR_O2, so the static matrix_o2 build is identically
# zero, per that diagnostic's own note) and is dropped from its amplitude --
# the 'mesospheric' row here is therefore OH-only flux, labelled accordingly
# below. log10 with a PER-GROUP data-driven floor (1% of that group's own
# median positive amplitude) keeps moon-down/near-zero rows from producing
# -inf without imposing one shared floor across groups whose amplitude scales
# differ by orders of magnitude.
#
# Row source: near + far sky-pointing decompositions pooled (their context
# columns are independent per pointing, so pooling roughly doubles coverage
# without literally duplicating rows). INCLUDE_SCI additionally pools in the
# science pointing.
#
# Column scope: CONTEXT_SCOPE switches between a curated, physically-central
# subset (default -- keeps the panel count sane while we iterate on layout)
# and the full final context set (all ~30 columns after decoding cyclic
# sin/cos pairs to single angles).
#
# Rendered as hexbin density per panel: pooled near+far is ~29k points/panel
# (more with INCLUDE_SCI), and a plain scatter would be unreadable without an
# alpha/marker-size compromise that hexbin avoids entirely.

import base64
import io

import matplotlib.pyplot as plt
from astropy.io import fits
from IPython.display import HTML as _HTML_ctxplot

CONTEXT_SCOPE = 'curated'   # 'curated' or 'all'
INCLUDE_SCI = True          # also pool the science-pointing rows in

# The continuum GROUP row is replaced by its three individual diffuse species
# below (HO2, FeO, O2Ac): the group mean hid which species actually carries a
# context dependence, and they are known to differ (e.g. FeO/OH tracks moon
# brightness -- see feo-oh-cap-sizing memory -- unlike HO2/O2Ac).
_AMP_GROUPS = ['moon', 'zodi', 'mesospheric', 'ionospheric', 'atomic']
_DIFFUSE_SPECIES = ['HO2', 'FeO', 'O2Ac']

# Physically-central subset: pointing/moon/sun geometry, one van Rhijn factor,
# ecliptic coordinates, and local time of night. Skips the raw lunation-of-
# year/year-of-survey angles and the F10.7/Kp/ew space-weather columns as
# secondary for this figure; also drops sci_sep, zodi_log10_v and
# moon_signal_proxy (2026-09-25: not informative enough here to keep), and
# vanrhijn_95km/285km (2026-09-25: redundant with vanrhijn_87km -- the three
# are highly collinear, one is enough to show the van Rhijn dependence).
_CURATED_CONTEXT = [
    'alt', 'airmass', 'moon_sep', 'moon_alt', 'moon_phase', 'moon_fli',
    'sun_sep', 'sun_alt', 'obstime_day',
    'vanrhijn_87km',
    'ecl_beta_deg', 'ecl_lon',
]

# Panels for this column get their x-range clamped by masking the input data
# (not just ax.set_xlim -- that would crop a hexbin already binned over the
# full range, at the coarser resolution that range implied) to the given
# (lo, hi) before hexbinning, so each clamped column renders at full gridsize
# resolution within the range that matters.
_XRANGE_CLAMP_CTXPLOT = {'airmass': (1.0, 2.0), 'vanrhijn_87km': (1.0, 2.0)}

# Per-ROW y-axis floor: masks out the low tail (mostly the per-row log floor
# artifact and its immediate neighbourhood) BEFORE hexbinning, same reasoning
# as the x clamps above -- cropping via ax.set_ylim afterward would keep the
# bins computed over the full (wider) range, at coarser effective resolution
# for the region actually shown. Only a lower bound; the upper end stays
# whatever the (masked) data autoscales to.
# Reset to empty (2026-09-29): the flux-integrated amplitude sits on a
# different numeric scale per row than the old mean-of-coefficients proxy
# (each v_k folds in that template's own integrated area), so the previous
# hand-tuned floors no longer apply -- retune per row once the new ranges are
# visible.
_YMIN_CTXPLOT = {}
# 'mesospheric' is OH-only now that O2_b01 has dropped out of its flux
# amplitude (see the amplitude comment above), so its row needs its own
# display label instead of the group's ML name.
_ROW_DISPLAY_LABEL_CTXPLOT = {'mesospheric': 'OH'}

# obstime_day decodes to UTC time-of-day in degrees (0 = UTC midnight), which
# splits the night's data across the 0/360 wrap and leaves the empty daytime
# gap sitting in the MIDDLE of a plain [0, 360) axis. Converted here to hours
# from LOCAL (LCO) midnight, wrapped to (-12, 12]: the empty gap moves to the
# ends of the range instead, so the panel reads continuously from local
# evening, through midnight, to local morning -- one arc, no internal gap.
_LCO_LON_DEG_CTXPLOT = -70.68992  # Las Campanas Observatory (data._lco_earth_location)

# --- Restrict to the rows actually used for learning, THEN pool pointings --
required_ctxplot = ['compress_train_idx']
_missing_req_ctxplot = [k for k in required_ctxplot if k not in filtered_triplet]
if _missing_req_ctxplot:
    raise RuntimeError(
        'Run the compressor-split cell first (it sets compress_train_idx on '
        'filtered_triplet). Missing: ' + ', '.join(_missing_req_ctxplot))
_train_idx_ctxplot = np.asarray(filtered_triplet['compress_train_idx'], dtype=int)

_pool_ctxplot = [filtered_triplet['coef_near'][_train_idx_ctxplot],
                 filtered_triplet['coef_far'][_train_idx_ctxplot]]
_pool_ctx_ctxplot = [filtered_triplet['ctx_near'][_train_idx_ctxplot],
                     filtered_triplet['ctx_far'][_train_idx_ctxplot]]
if INCLUDE_SCI:
    _pool_ctxplot.append(filtered_triplet['coef_sci'][_train_idx_ctxplot])
    _pool_ctx_ctxplot.append(filtered_triplet['ctx_sci'][_train_idx_ctxplot])
_coef_stack_ctxplot = np.concatenate(_pool_ctxplot, axis=0)
_ctx_stack_ctxplot = np.concatenate(_pool_ctx_ctxplot, axis=0)
_ctx_names_raw_ctxplot = [str(n) for n in filtered_triplet['ctx_names']]

_ctx_names_disp_ctxplot, _ctx_disp_ctxplot = data._decode_cyclic_context(
    _ctx_names_raw_ctxplot, _ctx_stack_ctxplot)

# Rewrite obstime_day IN PLACE from "UTC time-of-day, degrees" to "hours from
# local (LCO) midnight" -- see the comment by _LCO_LON_DEG_CTXPLOT above.
if 'obstime_day' in _ctx_names_disp_ctxplot:
    _od_idx_ctxplot = _ctx_names_disp_ctxplot.index('obstime_day')
    _utc_deg_ctxplot = _ctx_disp_ctxplot[:, _od_idx_ctxplot].astype(np.float64)
    _local_centered_deg_ctxplot = (
        (_utc_deg_ctxplot + _LCO_LON_DEG_CTXPLOT + 180.0) % 360.0) - 180.0
    _ctx_disp_ctxplot[:, _od_idx_ctxplot] = (
        _local_centered_deg_ctxplot / 15.0).astype(_ctx_disp_ctxplot.dtype)

# --- Column selection --------------------------------------------------------
if CONTEXT_SCOPE == 'curated':
    _missing_ctx = [c for c in _CURATED_CONTEXT if c not in _ctx_names_disp_ctxplot]
    if _missing_ctx:
        print(f'  curated context columns not found (skipped): {_missing_ctx}')
    _cols_ctxplot = [c for c in _CURATED_CONTEXT if c in _ctx_names_disp_ctxplot]
elif CONTEXT_SCOPE == 'all':
    _cols_ctxplot = list(_ctx_names_disp_ctxplot)
else:
    raise ValueError("CONTEXT_SCOPE must be 'curated' or 'all'")
_col_idx_ctxplot = [_ctx_names_disp_ctxplot.index(c) for c in _cols_ctxplot]

# --- Flux-integrated amplitude basis -----------------------------------------
# One decomposer instance supplies every family's static per-coefficient
# template; its wavelength-summed rows are the v_k this cell needs. Built from
# the EVERY10 wave grid with a generic LSF (lsf_sigma=1.0) or, for a telluric
# corpus, the representative-row telluric bundle -- exactly
# amplitude_error_vs_ctx.py's own choice, so both cells' amplitudes are
# calibrated the same way.
from sky_decomp.lsf_surface_iterative import SkyDecompLSFSurfaceIterative as _SkyDecompLSF_ctxplot
from mlp_predictor.data import make_reconstruction_decomposer as _mk_decomp_ctxplot

with fits.open(f'{DECOMP_DATA_ROOT}/{DECOMP_STEM}_every10.fits') as _h_ctxplot:
    _w_ctxplot = np.asarray(_h_ctxplot['WAVE'].data, dtype=np.float64)
    _wave_basis_ctxplot = _w_ctxplot if _w_ctxplot.ndim == 1 else _w_ctxplot[0]

_basis_tel_ctxplot = None
if USE_TELLURIC_RECON:
    _basis_tel_ctxplot = data.make_telluric_row_lookup(
        cfg.data.input_fits_for_basis, verbose=False,
        decomp_suffix=cfg.data.decomp_suffix)(
            'sci', data.telluric_representative_row(cfg.data.input_fits_for_basis))
    _amp_model_ctxplot = _mk_decomp_ctxplot(
        _wave_basis_ctxplot, n_spline_knots=N_MOON_KNOTS,
        base_dir=data._infer_base_dir_for_reconstruction(),
        split_zodi=SPLIT_ZODI, n_zodi_spline_knots=N_ZODI_KNOTS,
        telluric=_basis_tel_ctxplot)
else:
    _amp_model_ctxplot = _SkyDecompLSF_ctxplot(
        _wave_basis_ctxplot, lsf_sigma=1.0, n_spline_knots=N_MOON_KNOTS,
        base_dir=data._infer_base_dir_for_reconstruction(),
        split_zodi=SPLIT_ZODI, n_zodi_spline_knots=N_ZODI_KNOTS)

# Per-coefficient-NAME -> v_k = template_k(lambda).sum(). Looked up by name,
# never by position: the decomposer's 'atom'/'orc' family split does not
# match this project's atomic/ionospheric ML grouping. O2_b01 is absent on
# purpose (see the amplitude comment above).
_name_to_v_ctxplot = {}
for _mat_ctxplot, _names_ctxplot in (
    (_amp_model_ctxplot.matrix_moon, _amp_model_ctxplot.moon_names),
    (_amp_model_ctxplot.matrix_zodi, _amp_model_ctxplot.zodi_names),
    (_amp_model_ctxplot.matrix_diffuse, _amp_model_ctxplot.diffuse_names),
    (_amp_model_ctxplot.matrix_atom, _amp_model_ctxplot.atom_names),
    (_amp_model_ctxplot.matrix_orc, _amp_model_ctxplot.orc_names),
):
    for _row_ctxplot, _name_ctxplot in zip(
            np.asarray(_mat_ctxplot, dtype=np.float64), _names_ctxplot):
        _name_to_v_ctxplot[str(_name_ctxplot).lower()] = float(_row_ctxplot.sum())
for _i_ctxplot in range(_amp_model_ctxplot.matrix_oh.shape[0]):
    _name_to_v_ctxplot[f'oh_{_i_ctxplot:03d}'] = float(np.asarray(
        _amp_model_ctxplot.matrix_oh[_i_ctxplot], dtype=np.float64).sum())

# --- Amplitude proxies: log10 flux-integrated amplitude ---------------------
# Row order: moon, zodi, the three diffuse species (replacing the old combined
# 'continuum' group mean), then the remaining line-family groups.
_coef_names_lower_ctxplot = [str(n).lower() for n in filtered_triplet['coef_names']]
_ROW_ORDER_CTXPLOT = (
    [('group', g) for g in ('moon', 'zodi')]
    + [('species', s) for s in _DIFFUSE_SPECIES]
    + [('group', g) for g in _AMP_GROUPS if g not in ('moon', 'zodi')]
)
_dropped_warned_ctxplot = set()


def _find_coef_col_ctxplot(name):
    lname = name.lower()
    matches = [i for i, n in enumerate(_coef_names_lower_ctxplot) if n == lname]
    if not matches:
        raise KeyError(f'coefficient {name!r} not in coef_names')
    return matches[0]


def _floor_log10_ctxplot(values):
    _pos = values[values > 0]
    _floor = max(1e-8, 0.01 * float(np.median(_pos))) if _pos.size >= 20 else 1e-8
    return np.log10(np.clip(values, _floor, None))


def _group_amplitude_ctxplot(label):
    _idx = group_indices.get(label)
    if _idx is None or len(_idx) == 0:
        return None
    _keep, _v = [], []
    _dropped = []
    for _j in _idx:
        _n = _coef_names_lower_ctxplot[_j]
        if _n in _name_to_v_ctxplot:
            _keep.append(_j)
            _v.append(_name_to_v_ctxplot[_n])
        else:
            _dropped.append(_n)
    if _dropped and label not in _dropped_warned_ctxplot:
        print(f'  {label}: no static template for {_dropped} (per-row-fit '
              'template); excluded from its flux amplitude.')
        _dropped_warned_ctxplot.add(label)
    _v = np.asarray(_v, dtype=np.float64)
    return _coef_stack_ctxplot[:, _keep] @ _v


def _species_amplitude_ctxplot(name):
    _ci = _find_coef_col_ctxplot(name)
    return _coef_stack_ctxplot[:, _ci] * _name_to_v_ctxplot[name.lower()]


_amp_rows_ctxplot = {}
_amp_is_species_ctxplot = set()  # single named coefficient, not a group mean
for _kind, _label in _ROW_ORDER_CTXPLOT:
    if _kind == 'group':
        _values = _group_amplitude_ctxplot(_label)
        if _values is None:
            continue
    else:
        _values = _species_amplitude_ctxplot(_label)
        _amp_is_species_ctxplot.add(_label)
    _amp_rows_ctxplot[_label] = _floor_log10_ctxplot(_values)

_row_labels_ctxplot = list(_amp_rows_ctxplot)
n_rows_ctxplot = len(_row_labels_ctxplot)
n_cols_ctxplot = len(_cols_ctxplot)
print(f'Context-vs-amplitude grid: {n_rows_ctxplot} rows x {n_cols_ctxplot} cols, '
      f'{_coef_stack_ctxplot.shape[0]} pooled training rows '
      f"({'near+far+sci' if INCLUDE_SCI else 'near+far'}, "
      f'{_train_idx_ctxplot.size} of {filtered_triplet["coef_near"].shape[0]} '
      f'filtered rows used for learning).')

# --- Render: hexbin density per panel ---------------------------------------
# Panel size bumped alongside the larger font below: matplotlib doesn't grow
# a fixed-size axes to fit bigger tick/label text, so more canvas is needed
# or the larger text just crowds/overlaps between panels.
_panel_w_in, _panel_h_in = 2.6, 2.2
_fig_w_ctxplot = _panel_w_in * n_cols_ctxplot + 1.4
_fig_h_ctxplot = _panel_h_in * n_rows_ctxplot + 1.2

fig_ctxplot, axes_ctxplot = plt.subplots(
    n_rows_ctxplot, n_cols_ctxplot, sharex='col', sharey='row',
    figsize=(_fig_w_ctxplot, _fig_h_ctxplot), squeeze=False)

# obstime_day no longer means "time of day in degrees" after the local-hour
# rewrite above, so it needs its own title/xlabel instead of the generic
# underscore-to-space rule.
_COL_DISPLAY_LABEL_CTXPLOT = {'obstime_day': 'local hour (0 = midnight)'}

for r_i, _g in enumerate(_row_labels_ctxplot):
    _y = _amp_rows_ctxplot[_g]
    _ymin_ctxplot = _YMIN_CTXPLOT.get(_g)
    for c_i, _cidx in enumerate(_col_idx_ctxplot):
        ax = axes_ctxplot[r_i, c_i]
        _col_name_ctxplot = _cols_ctxplot[c_i]
        _x = _ctx_disp_ctxplot[:, _cidx]
        _m = np.isfinite(_x) & np.isfinite(_y)
        if _ymin_ctxplot is not None:
            _m &= (_y >= _ymin_ctxplot)
        _clamp = _XRANGE_CLAMP_CTXPLOT.get(_col_name_ctxplot)
        if _clamp is not None:
            _m &= (_x >= _clamp[0]) & (_x <= _clamp[1])
        # Greys: white = zero/low density, black = high density. mincnt=1
        # already leaves zero-count hexagons undrawn, so the white figure
        # background IS the zero level rather than the palette's pale end.
        ax.hexbin(_x[_m], _y[_m], gridsize=25, cmap='Greys', mincnt=1,
                  linewidths=0.1, rasterized=True)
        ax.tick_params(axis='both', labelsize=9, length=3, pad=2)
        for _sp in ax.spines.values():
            _sp.set_linewidth(0.5)
        _col_label_ctxplot = _COL_DISPLAY_LABEL_CTXPLOT.get(
            _col_name_ctxplot, _col_name_ctxplot.replace('_', ' '))
        if r_i == 0:
            ax.set_title(_col_label_ctxplot, fontsize=14)
        if r_i == n_rows_ctxplot - 1:
            ax.set_xlabel(_col_label_ctxplot, fontsize=14)
            ax.tick_params(axis='x', labelrotation=45)
        if c_i == 0:
            _g_disp = _ROW_DISPLAY_LABEL_CTXPLOT.get(_g, _g)
            _ylab = _g_disp if _g in _amp_is_species_ctxplot else f'A({_g_disp})'
            ax.set_ylabel(f'log {_ylab}', fontsize=14)

# fig_ctxplot.suptitle(
#     f'Decomposition amplitude vs context '
#     f"({'near+far+sci' if INCLUDE_SCI else 'near+far'} pooled, training rows only, "
#     f'n={_coef_stack_ctxplot.shape[0]}; hexbin density, log10 y)',
#     fontsize=15, y=1.0)
fig_ctxplot.subplots_adjust(left=0.045, right=0.995, top=0.93, bottom=0.09,
                            wspace=0.08, hspace=0.18)

fig_ctxplot.savefig(FIGDIR / 'context_vs_amplitude.pdf')
print(f"Saved {FIGDIR / 'context_vs_amplitude.pdf'}")

# Inline preview: horizontal-scroll wrapper (VS Code caps inline image width
# at the cell width regardless of Image(width=...), same workaround as the
# old relationship_scatter_matrix.py cell). Also keeps a bounded-size PNG
# separate from the full vector PDF saved above.
_buf_ctxplot = io.BytesIO()
fig_ctxplot.savefig(_buf_ctxplot, format='png', dpi=110, bbox_inches='tight')
plt.close(fig_ctxplot)
_buf_ctxplot.seek(0)
_png_ctxplot = _buf_ctxplot.getvalue()
_b64_ctxplot = base64.b64encode(_png_ctxplot).decode('ascii')
_display_px_ctxplot = int(_fig_w_ctxplot * 70)
display(_HTML_ctxplot(
    f'<div style="overflow-x: auto; overflow-y: hidden; width: 100%;">'
    f'<img src="data:image/png;base64,{_b64_ctxplot}" '
    f'style="width: {_display_px_ctxplot}px; max-width: none; height: auto; '
    f'display: block;" /></div>'))





In [ ]:
# --- Example decomposition: observed vs reconstruction (moon up / moon down)
# Starting point: mlp_predictor/diagnostics_cells/full_spectrum_single_row.py
# and the cell below that calls it (diag.full_spectrum_single_row) -- but this
# only needs the DECOMPOSITION's own fit to one sky channel's own spectrum, not
# the ML transfer that diagnostic also shows. No mlp_artifacts, no context
# augmentation: reconstruct_with_lsf runs on the channel's OWN fitted
# coefficients, so this cell is self-contained (needs only cell 0's globals
# and the wavelengths cell's N_MOON_KNOTS/N_ZODI_KNOTS).
#
# Row selection: within the every10 stack (the stride-10 subsample that keeps
# full per-pixel flux, the same file every single-row diagnostic in this
# codebase indexes into -- the full corpus's coefficient tables don't carry
# per-pixel flux), pick one moon-up and one moon-down example whose CHANNEL
# chi2 sits at GOOD_CHI2_PERCENTILE within its moon-state bucket: a
# representative good fit, not literally the single best (which risks a
# near-empty or otherwise degenerate spectrum).
import numpy as np
from astropy.io import fits
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import decompose_parallel

CHANNEL = 'near'              # 'near', 'far', or 'sci' -- which pointing's own decomposition to show
GOOD_CHI2_PERCENTILE = 25.0    # row selection target within its moon-state bucket
MOON_UP_ALT_MIN = 20.0         # deg; well up, not a marginal/twilight case
MOON_UP_FLI_MIN = 0.3          # illuminated fraction; visibly bright for the "up" example

_texfont_ex = 'CMU Serif, Latin Modern Roman, Times New Roman, Times, serif'
# (key, LaTeX legend label, color) -- key stays plain text for dict lookups;
# the label is emitted through MathJax, so it is the one place allowed to
# carry raw LaTeX markup like the OH $_2$ subscript.
_COMPONENTS_EX = [
    ('moonlight', r'\text{moonlight}', 'blue', 0.9),
    ('zodi', r'\text{zodiacal light}', 'purple', 0.9),
    ('oh_o2', r'\text{OH + O}_2\text{ airglow}', '#8f8f8f', 0.5),
    ('diffuse', r'\text{diffuse continuum}', 'orange', 0.9),
    ('atomic', r'\text{atomic + ionospheric lines}', 'dimgray', 0.9),
]
# Draw order (z-order: earlier = further back) is independent of the list
# above, which instead fixes the legend grouping/order. OH and atomic first
# so the smoother continuum curves and the observed/total curves all sit on
# top of the two line forests, which is what makes those forests readable as
# a texture rather than a solid block obscuring everything under them.
_DRAW_ORDER_EX = ['oh_o2', 'atomic'] + [
    k for k, *_ in _COMPONENTS_EX if k not in ('oh_o2', 'atomic')]

_e10_stem_ex = f"{DECOMP_DATA_ROOT}/{DECOMP_STEM}_every10"
EVERY10_INPUT_EX = f"{_e10_stem_ex}.fits"
EVERY10_DECOMP_EX = {
    'near': f"{_e10_stem_ex}_decomp_sky1{cfg.data.decomp_suffix}.fits",
    'far':  f"{_e10_stem_ex}_decomp_sky2{cfg.data.decomp_suffix}.fits",
    'sci':  f"{_e10_stem_ex}_decomp_sci{cfg.data.decomp_suffix}.fits",
}
_FLUX_KEY_EX = {'near': 'FLUX_SKY_NEAR', 'far': 'FLUX_SKY_FAR', 'sci': 'FLUX_SCI'}[CHANNEL]
_KIND_MAP_EX = {'near': 'sky1', 'far': 'sky2', 'sci': 'sci'}[CHANNEL]

# moon_phase_cos is the raw (un-decoded) cos of the lunar phase angle, so the
# illuminated fraction FLI = (1 - cos(phase))/2 (0 = new, 1 = full) needs no
# cyclic decoding -- avoids pulling in the full ctx-augmentation pipeline that
# only the ML transfer path needs.
e10_triplet_ex = data.build_triplet_coef_dataset(
    input_fits_path=EVERY10_INPUT_EX,
    sky_near_decomp_fits_path=EVERY10_DECOMP_EX['near'],
    sky_far_decomp_fits_path=EVERY10_DECOMP_EX['far'],
    sci_decomp_fits_path=EVERY10_DECOMP_EX['sci'],
    context_columns=['moon_alt', 'moon_phase_cos'],
    return_chi2=True,
)
_ctx_names_ex = list(e10_triplet_ex['ctx_names'])
_moon_alt_ex = np.asarray(
    e10_triplet_ex[f'ctx_{CHANNEL}'][:, _ctx_names_ex.index('moon_alt')], dtype=np.float64)
_moon_fli_ex = (1.0 - np.asarray(
    e10_triplet_ex[f'ctx_{CHANNEL}'][:, _ctx_names_ex.index('moon_phase_cos')],
    dtype=np.float64)) / 2.0
_chi2_ex = np.asarray(e10_triplet_ex[f'chi2_{CHANNEL}'], dtype=np.float64)


def _pick_example_row_ex(mask, percentile):
    idx = np.flatnonzero(mask & np.isfinite(_chi2_ex))
    if idx.size == 0:
        raise RuntimeError('No every10 rows satisfy the selection mask.')
    target = np.percentile(_chi2_ex[idx], percentile)
    return int(idx[np.argmin(np.abs(_chi2_ex[idx] - target))])


_pos_up_ex = _pick_example_row_ex(
    (_moon_alt_ex > MOON_UP_ALT_MIN) & (_moon_fli_ex > MOON_UP_FLI_MIN), GOOD_CHI2_PERCENTILE)
_pos_down_ex = _pick_example_row_ex(_moon_alt_ex <= 0.0, GOOD_CHI2_PERCENTILE)

with fits.open(EVERY10_INPUT_EX) as _hdul_ex:
    _wave_arr_ex = np.asarray(_hdul_ex['WAVE'].data, dtype=np.float64)
    _flux_all_ex = np.asarray(_hdul_ex[_FLUX_KEY_EX].data, dtype=np.float64)
    _lsf_sci_arr_ex = np.asarray(_hdul_ex['LSF_SCI'].data, dtype=np.float64)

_base_dir_ex = data._infer_base_dir_for_reconstruction()
_tel_lookup_ex = (
    data.make_telluric_row_lookup(cfg.data.input_fits_for_basis,
                                  decomp_suffix=cfg.data.decomp_suffix)
    if USE_TELLURIC_RECON else None)


def _build_example_ex(pos):
    idx_row = int(e10_triplet_ex['row_index'][pos])
    wave_row = _wave_arr_ex if _wave_arr_ex.ndim == 1 else _wave_arr_ex[idx_row]
    lsf_row = _lsf_sci_arr_ex if _lsf_sci_arr_ex.ndim == 1 else _lsf_sci_arr_ex[idx_row]
    flux_row = _flux_all_ex[idx_row]
    coef_row = np.asarray(e10_triplet_ex[f'coef_{CHANNEL}'][pos], dtype=np.float64)
    lsf_state = data.load_lsf_state_if_available(EVERY10_DECOMP_EX[CHANNEL], idx_row)
    o2_vec = data.load_o2_vector_if_available(EVERY10_DECOMP_EX[CHANNEL], idx_row)
    tel = None if _tel_lookup_ex is None else _tel_lookup_ex(_KIND_MAP_EX, idx_row)
    comps = data.reconstruct_with_lsf(
        wave=wave_row, coef=coef_row,
        lsf=lsf_state if lsf_state is not None else (lsf_row / 2.35),
        n_spline_knots=N_MOON_KNOTS, n_zodi_spline_knots=N_ZODI_KNOTS,
        base_dir=_base_dir_ex, o2_vector=o2_vec, telluric=tel,
    )
    flux_recon_row = np.asarray(comps['total'], dtype=np.float64) / FACTOR
    return dict(
        wave=wave_row, flux=flux_row, comps=comps, lsf_fwhm=lsf_row,
        resid=flux_row - flux_recon_row,
        chi2=float(_chi2_ex[pos]), moon_alt=float(_moon_alt_ex[pos]),
        moon_fli=float(_moon_fli_ex[pos]), idx_row=idx_row,
    )


example_up = _build_example_ex(_pos_up_ex)
example_down = _build_example_ex(_pos_down_ex)
for _label, _ex in (('moon-up', example_up), ('moon-down', example_down)):
    print(f'{_label} example ({CHANNEL}): every10 row {_ex["idx_row"]}, '
          f'chi2={_ex["chi2"]:.4g}, moon_alt={_ex["moon_alt"]:.1f} deg, '
          f'moon_fli={_ex["moon_fli"]:.2f}')


def _mask_to_spans_ex(wave, mask):
    """Contiguous True runs of a boolean mask, as (wave_lo, wave_hi) pairs."""
    spans = []
    start = None
    for i, m in enumerate(mask):
        if m and start is None:
            start = wave[i]
        elif not m and start is not None:
            spans.append((start, wave[i - 1]))
            start = None
    if start is not None:
        spans.append((start, wave[-1]))
    return spans


# Zoom windows shown as their own small panels above the main flux panel,
# each a closer look at one line-forest region: 3900-4200 A (blue OH band
# edge), 6200-6500 A (Halpha/[NII] region, also science-line-masked), and
# 8400-8700 A (red OH band).
_ZOOM_RANGES_EX = [(3900.0, 4200.0), (6200.0, 6500.0), (8400.0, 8700.0)]

# Grid: row 1 = the three zoom panels (one column each); row 2 = the main
# flux panel; row 3 = the residual panel. Rows 2-3 each span all 3 columns
# and share an x-axis (wired up below via `matches`, since make_subplots'
# own `shared_xaxes` links axes column-by-column and would incorrectly tie
# the zoom panels' x-axes to the main panel's).
_GRID_SPECS_EX = [
    [{}, {}, {}],
    [{"colspan": 3}, None, None],
    [{"colspan": 3}, None, None],
]
_GRID_ROW_HEIGHTS_EX = [0.26/0.8, 0.52/0.8, 0.22/0.8]


def _make_example_figure_ex(ex):
    _comps = ex['comps']
    _species_ex = {
        'moonlight': _comps['moon'],
        'zodi': _comps.get('zodi', np.zeros_like(_comps['moon'])),
        'oh_o2': _comps['oh'] + _comps['o2'],
        'diffuse': _comps['diffuse'],
        'atomic': _comps['atom'] + _comps['orc'],
    }
    fig = make_subplots(rows=3, cols=3, specs=_GRID_SPECS_EX,
                         row_heights=_GRID_ROW_HEIGHTS_EX,
                         vertical_spacing=0.08, horizontal_spacing=0.06)
    # Log axis + exact/near-zero component values (a line-family component is
    # 0 between its own lines by construction) is a bad mix: Plotly does not
    # just drop non-positive points, it autoscales to whatever tiny positive
    # float is there (down to 1e-24 on faint Gaussian wings), crushing the
    # entire physically-relevant range into a sliver at the top. Floor every
    # flux-panel curve well below the faintest real continuum feature instead.
    _LOG_FLOOR_EX = 1e-2  # display units (1e-16 erg/s/cm2/A)

    def _floored_ex(y):
        return np.clip(np.asarray(y, dtype=np.float64), _LOG_FLOOR_EX, None)

    # Two-column legend: two independent Plotly legends side by side (paper
    # coords, positioned in update_layout below) instead of one tall single
    # column -- there is no native "N columns" legend property in Plotly.
    # Left column: observed + the first half of _COMPONENTS_EX (its own,
    # legend-only order); right column: the rest + total reconstruction.
    # This grouping is independent of _DRAW_ORDER_EX, the z-order below.
    _n_left_ex = (len(_COMPONENTS_EX) + 1) // 2  # +1 to balance against "observed"
    _legend_group_ex = {
        _key: ('legend' if _i < _n_left_ex else 'legend2')
        for _i, (_key, *_rest) in enumerate(_COMPONENTS_EX)}
    _comp_by_key_ex = {_key: (_label, _color, _width)
                       for _key, _label, _color, _width in _COMPONENTS_EX}

    def _add_component_trace_ex(_key, row, col, showlegend):
        _label, _color, _width = _comp_by_key_ex[_key]
        fig.add_trace(go.Scattergl(
            x=ex['wave'], y=_floored_ex(_species_ex[_key]), mode='lines',
            name=f'${_label}$', line=dict(color=_color, width=_width),
            legend=_legend_group_ex[_key] if showlegend else None,
            showlegend=showlegend), row=row, col=col)

    def _add_flux_panel_ex(row, col, showlegend):
        # oh_o2 and atomic first (bottom layer): the two line forests, drawn
        # under everything else -- including observed -- so the smoother
        # continuum curves and observed/total read cleanly on top of them.
        _add_component_trace_ex('oh_o2', row, col, showlegend)
        _add_component_trace_ex('atomic', row, col, showlegend)
        fig.add_trace(go.Scattergl(
            x=ex['wave'], y=_floored_ex(ex['flux'] * FACTOR), mode='lines',
            name=r'$\text{observed}$', line=dict(color='black', width=0.8),
            legend='legend' if showlegend else None, showlegend=showlegend),
            row=row, col=col)
        for _key in _DRAW_ORDER_EX[2:]:
            _add_component_trace_ex(_key, row, col, showlegend)
        fig.add_trace(go.Scattergl(
            x=ex['wave'], y=_floored_ex(_comps['total']), mode='lines',
            name=r'$\text{total reconstruction}$', line=dict(color='red', width=1.0),
            legend='legend2' if showlegend else None, showlegend=showlegend),
            row=row, col=col)

    for _col, _ in enumerate(_ZOOM_RANGES_EX, start=1):
        _add_flux_panel_ex(row=1, col=_col, showlegend=False)
    _add_flux_panel_ex(row=2, col=1, showlegend=True)

    fig.add_trace(go.Scattergl(
        x=ex['wave'], y=ex['resid'] * FACTOR, mode='lines', showlegend=False,
        line=dict(color='black', width=0.6)), row=3, col=1)
    fig.add_hline(y=0, line=dict(color='gray', width=0.6, dash='dash'), row=3, col=1)
    # Science emission-line mask (decompose_parallel.science_line_mask):
    # rest-frame windows -- this cell does not measure the per-row Halpha
    # velocity decompose_parallel centres them on -- around the ten nebular
    # lines that get IVAR=0 in every arm at fit time, since none of them is in
    # the 388-component sky basis (see decompose_parallel.py's "Science
    # emission-line mask" section). Marked here as a gray band so a reader
    # does not mistake the resulting small OH/moon/zodi bias at those
    # wavelengths for a fit failure. Added AFTER the traces above: add_vrect
    # silently drops shapes on a still-empty subplot (exclude_empty_subplots
    # defaults to True), so calling it before any trace exists is a no-op.
    _line_mask_ex, _ = decompose_parallel.science_line_mask(
        ex['wave'], lsf_fwhm=ex['lsf_fwhm'])
    _vrect_targets_ex = [(1, 1), (1, 2), (1, 3), (2, 1), (3, 1)]
    for _lo, _hi in _mask_to_spans_ex(ex['wave'], _line_mask_ex):
        for _row, _col in _vrect_targets_ex:
            fig.add_vrect(x0=_lo, x1=_hi, row=_row, col=_col, fillcolor='dimgray',
                         opacity=0.5, line_width=0, layer='below')
    # MathJax's TeX input has no \AA macro outside a full LaTeX preamble --
    # the literal Angstrom character renders correctly inside \text{}, the
    # macro does not.
    fig.update_xaxes(title_text=r'$\text{Wavelength (Å)}$', tickfont=dict(size=16),
                     row=3, col=1)
    # Tick labels hidden here (not just matched range): row 3 already shows
    # the same wavelength axis directly below, joined tight against this one
    # (see the domain override below), so repeating them would be redundant.
    fig.update_xaxes(showticklabels=False, row=2, col=1)
    fig.update_yaxes(
        title_text=r'$f_\lambda\ (10^{-14}\,\mathrm{erg\,s^{-1}\,cm^{-2}\,\text{Å}^{-1}})$',
        type='log', tickfont=dict(size=16), row=2, col=1)
    fig.update_yaxes(title_text=r'$\text{Residual}$', tickfont=dict(size=16),
                     range=[-1, 4], row=3, col=1)
    # Row 3 (residual) shares its x-axis with row 2 (main flux) -- pan/zoom
    # stays in sync -- via `matches`, not make_subplots' shared_xaxes: that
    # links axes column-by-column and would incorrectly tie the zoom panels'
    # x-axes (row 1) to the main panel's too.
    _main_xaxis_id_ex = fig.get_subplot(row=2, col=1).xaxis.plotly_name.replace(
        'axis', '')
    fig.update_xaxes(matches=_main_xaxis_id_ex, row=3, col=1)
    # Close the visual gap between the main and residual panels so the joined
    # x-axis above reads as one shared boundary instead of two panels with a
    # tick-labelled main panel floating above a separate residual panel.
    # make_subplots' own `vertical_spacing` is one uniform gap for every row
    # pair, so the zoom-row gap is left alone and only rows 2-3 are pulled
    # together, by giving row 3 back the reclaimed space instead of shrinking
    # it.
    _main_y_lo_ex = fig.get_subplot(row=2, col=1).yaxis.domain[0]
    _resid_y_lo_ex = fig.get_subplot(row=3, col=1).yaxis.domain[0]
    fig.update_yaxes(domain=[_resid_y_lo_ex, _main_y_lo_ex - 0.015], row=3, col=1)
    # Zoom panels: each its own wavelength window, but the SAME y-range/scale
    # as the main panel (via matches) so peak heights are directly
    # comparable, and no per-panel axis titles -- tick labels alone make the
    # zoomed range self-evident, and the main/residual panels already carry
    # the shared title text.
    _main_yaxis_id_ex = fig.get_subplot(row=2, col=1).yaxis.plotly_name.replace(
        'axis', '')
    for _col, (_lo, _hi) in enumerate(_ZOOM_RANGES_EX, start=1):
        fig.update_xaxes(range=[_lo, _hi], tickfont=dict(size=12), row=1, col=_col)
        fig.update_yaxes(type='log', matches=_main_yaxis_id_ex,
                         tickfont=dict(size=12), row=1, col=_col)
    # Legend y-position: top of the MAIN panel's own y-domain, not the
    # figure's y=0.98 -- that used to be the flux panel's top edge when it
    # was row 1 of 2, but the zoom row now occupies the figure's actual top.
    _main_y_top_ex = fig.get_subplot(row=2, col=1).yaxis.domain[1] - 0.01
    _legend_style_ex = dict(bgcolor='rgba(255,255,255,0.7)', y=_main_y_top_ex,
                            yanchor='top', font=dict(size=11))
    fig.update_layout(
        height=700, width=825, font=dict(family=_texfont_ex, size=16),
        margin=dict(t=20, b=60),
        # Two legends side by side (paper coords, both top-left-ish) stand in
        # for a two-column legend -- see the comment above the traces.
        legend=dict(x=0.02, xanchor='left', **_legend_style_ex),
        legend2=dict(x=0.22, xanchor='left', **_legend_style_ex))
    return fig

fig_example_up = _make_example_figure_ex(example_up)
fig_example_up = show_with_pdf_export(fig_example_up, FIGDIR / 'example_decomp_moonup.pdf')

fig_example_down = _make_example_figure_ex(example_down)
fig_example_down = show_with_pdf_export(fig_example_down, FIGDIR / 'example_decomp_moondown.pdf')



In [ ]:
# --- Training config (edit here to override deployed defaults) ---
# Full deployed config expanded in place so every knob is visible; every value
# is initialised to the mlp_predictor.trainer default and can be changed below.

# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# 2026-08-27c: Phase F -- additive moon-zodi coupling.  Adds a small
# coupling branch (~9k params) that produces a shared latent from the
# moon-zodi ctx union, projected additively into the moon head output
# and the zodi head output via zero-init linear projectors.  Both
# existing paths (moon on shared trunk, zodi on isolated branch) are
# preserved -- this is the "preserve zodi isolation, give moon the
# shared context" fix for Phase E's atlas-mid regression (2026-08-27b).
# At init the projector outputs are zero, so training starts byte-
# identical to Phase A''; the projectors learn to route the coupling.
# To A/B against Phase A'' (no coupling), set
# "moon_zodi_coupling_enabled": False -- or ablations.apply(train_cfg,
# "A1_no_coupling").  The old instruction ("set the four moon_zodi_* entries
# to None") stopped working when the 2026-08-27 trim deleted the no-coupling
# path: it raised on an empty restriction instead of disabling anything.
# Phase E (moon_zodi_mode="shared_branch") is gone from model.py entirely.
# 2026-08-26f: Phase A' landed on top of Phase D.  Adds three explicit
# interaction features to the ctx augment pipeline (computed in
# data._augment_triplet_with_physics_priors):
#   moon_fli_x_phase_cos = moon_fli * moon_phase_cos          # 2nd-order phase
#   moon_sig_x_lon_cos   = moon_signal_proxy * ecl_lon_cos    # moon x zodi geom
#   moon_sig_x_lon_sin   = moon_signal_proxy * ecl_lon_sin    # moon x zodi geom
# All three fed into the isolated continuum branch (see continuum_ctx_restriction
# below).  Attacks the residual_ctx_attribution RF top importances on continuum
# (moon_fli 0.21, moon_phase_cos 0.15, ecl_lon_cos 0.09, moon_signal_proxy 0.05).
# A/B vs 26e (Phase D, same 10 seeds, n_test=1231):
#   mean_eRMSE  21.12 -> 20.98 (-0.7%)  |  mean_eWRMSE 14.98 -> 14.71 (-1.8%)
#   seed std    0.801 -> 0.509 (-36%, most of Phase-D variance recovered)
#   sky-arm continuum p50 err/Delta: all 1.19 -> 1.16, moon_down 1.10 -> 1.08,
#     close_zodi 1.27 -> 1.32 (only regression; still << pre-Phase-D 1.40).
#   wavelength_residual_atlas blue RMS|frac|: 0.535% -> 0.209% (-61%, huge).
#     mid +11% (still 0.63%), NIR +1.5%.  Mean bias grew to -0.4% mid / -0.6% NIR.
#   naive_baseline continuum ML now beats B1_near_geo in every regime (first
#     time): all +2.0%, moon_up +0.2%, moon_down +7.1%, close_zodi +2.4%.
# To A/B against pre-Phase-A' (Phase D only), drop the three trailing entries
# from `continuum_ctx_restriction` below.  The augment always emits them; they
# are simply ignored by the branch when not listed.
# ----------------------------------------------------------------------
# 2026-08-26e: Phase D landed on top of Phase A+B+C.  Widens the isolated
# continuum branch from (64, 32) -> (128, 64) and adds a 64-d hidden layer
# inside the continuum head (+12k targeted params, +2.3% full-model params).
# A/B vs 26d (Phase A+B+C, same 10 seeds, n_test=1231):
#   sky-arm floor, continuum p50 err/Delta:
#     all        1.247 -> 1.186  (-5%)
#     moon_down  1.212 -> 1.095  (-10%, target hit)
#     close_zodi 1.401 -> 1.274  (-9%,  target hit)
#   naive_baseline continuum sRMSE all: 0.01554 -> 0.01461 (-6%, recovers
#     the Phase-B accounting regression; ML now also beats B1_near_geo on
#     continuum moon_down for the first time).
#   wavelength_residual_atlas: blue -12%, NIR -5%, mid +5% (still zero-bias).
#   Aggregate mean_eRMSE 20.99 -> 21.12 (within seed noise); seed std
#     0.41 -> 0.80 (+96%, variance-inflation from the extra 12k params;
#     ensemble stderr still 1.2% of mean).
# To A/B against the pre-Phase-D config (Phase A+B+C only), override:
#     "continuum_branch_dims":     (64, 32)  # current: (128, 64)
#     "continuum_head_extra_dims": ()        # current: (64,)
# To revert further to the pre-Phase-A/B/C baseline, additionally override:
#     "moon_group_weight":         4.0       # current: 2.0
#     "continuum_group_weight":    1.5       # current: 1.0
#     "zodi_head_extra_dims":      ()        # current: (32,)
#     "continuum_ctx_restriction": None      # current: 6-tuple moon-geom
#     "alpha_ctx_features":        None      # current: 3-tuple
#     "alpha_ctx_groups":          None      # current: (moon, zodi, continuum)
# See 2026-08-26d + 2026-08-26e changelog entries for the full A/B tables.
# ----------------------------------------------------------------------
train_cfg = {
    "name": "dual_group_mlp_compressed",

    # Optimisation schedule.
    # 2026-09-18: 50 -> 300 epochs, patience 12 -> 100.  Early stopping does not
    # protect quality here (the loop restores `best_state`), so patience only
    # saved compute -- and at 12 it fired inside the plateau noise, stopping two
    # of four seeds at epoch 17 and 24 with models from epoch 5 and 12.  The
    # stall before the eventual best epoch is 28-131 epochs.  Measured in FLUX
    # space, 300 epochs beats 50 by -8.9% fractional RMS on the reconstructed
    # sky (-13.8% blue, -12.0% integrated), better on 4/4 paired seeds; the
    # coefficient metrics say the opposite because 92% of mean_eRMSE is the
    # mesospheric block.  See trainer.default_dual_group_config for the
    # patience simulation table.
    "n_epochs": 400,
    "batch_size": 512,
    "lr": 1.0e-3,
    "weight_decay": 2.0e-4,
    "patience": 100,

    # Architecture (widths).
    "encoder_dims": (768, 384),
    "ctx_dims": (96,),
    "trunk_dims": (320, 160),
    "head_dim": 192,
    "zodi_head_extra_dims": (),   # A4 (2026-09-04): was (32,).  Ablation showed the 7461-param
                                #  extra layer buys nothing measurable -- every group
                                #  within +/-1pp, aggregate -0.144 (inside the 0.28 seed
                                #  std).  The zodi head sits at err/delta = 0.354, far
                                #  inside its irreducible band, so extra capacity there
                                #  only chases noise.

    # Per-group loss weights m_g (see §3.6.2).
    "moon_group_weight": 3.0,
    "zodi_group_weight": 2.0,
    "continuum_group_weight": 1.0,
    "mesospheric_group_weight": 1.0,
    "ionospheric_group_weight": 1.0,

    # Loss shaping.
    # ADOPTED 2026-09-04: continuum joins moon and zodi.  Until this date the
    # diffuse basis was never wired into _precompute_flux_basis_and_geometry,
    # so naming "continuum" here silently did nothing -- and with no
    # flux-space term the group simply inherited the naive baseline's scale
    # error: signed flux bias -4.21% against B0_copy_near's -4.14%, while its
    # per-coefficient calibration looked fine (the three diffuse basis
    # functions have very different flux integrals, so the mean coefficient
    # is not the flux-weighted bias).
    # Replicated on two 10-seed blocks: continuum gain +8.6pp / +11.4pp,
    # bias -4.21% -> -0.43% and -4.03% -> -0.79%.  Every other group is
    # neutral or better in both blocks and mean_eRMSE improves ~0.04.
    "flux_mse_groups": ("moon", "zodi", "continuum"),
    # Additive log-amplitude term in flux space (§3.6.1).  0.0 = off, the
    # deployed default.  The per-pixel flux term above is ABSOLUTE and so is
    # dominated by the brightest rows; this one is scale-free and targets the
    # measured tail, 83-89% of whose MSE is pure brightness.  lambda must be
    # O(1-10) to matter -- log**2 is ~0.01 for a 0.1 dex miss while the
    # per-pixel term is rescaled to the coefficient loss's magnitude.
    # ADOPTED 2026-09-04: moon only.  Replicated on two independent 10-seed
    # blocks -- moon relative integrated-amplitude error -20.3% / -18.6%,
    # moon core flux RMSE -2.3% / -1.4%, continuum +1.4pp / +1.6pp.
    # Costs mesospheric ~-0.7pp and mean_eRMSE +0.05..+0.09: that aggregate
    # is dominated by the 358 mesospheric coefficients (scale ~34), not by
    # moon and zodi (scale ~0.5), so it moves against the groups this helps.
    # ZODI IS DELIBERATELY 0.  A global lambda drives the zodi gain from
    # +38.3% to +21.8% (lambda 20): the zodi amplitude is already pinned by
    # the decomposition's absolute Leinert anchor -- re-measured 2026-09-09 on
    # gaia-stars-mask with the exact per-row design, 87.5% of moon-up rows sit
    # exactly on the kappa_z ceiling -- so penalising it harder only trades
    # colour for amplitude the network already had.
    # That ceiling is CORRECT and should not be widened: releasing it on 294
    # lunation-stratified spectra moves the freed zodi up 0.193 dex while the
    # moon falls 0.107 and the diffuse block falls 0.435, with the total
    # continuum conserved to 0.5% and rms improving only 0.6%.  The excess
    # tracks the MOON, not the ecliptic -- partial rho(excess, FLI | B500) =
    # +0.733 against partial rho(excess, B500 | FLI) = -0.322, which has the
    # wrong sign for zodiacal light.  So on bright-moon rows the zodi target
    # is a constraint, exactly as the moon target is on moon-down rows, and
    # the matching fix is a ceiling RULE (needs the physics-only Z_pred in the
    # moon/zodi cache, which v1 does not carry -- it stores the
    # learned-parameter variant).
    # lambda 5 is near-optimal; moon-only 20 and 50 give +18.2% and +9.7%
    # moon gain against +24.2% at 5.  See ablations.describe("S2a_flux_amp_1").
    "flux_amp_lambda": {"moon": 5.0, "zodi": 0.0},
    "flux_amp_floor_frac": 0.05,   # eps = 5% of the group median amplitude

    # Per-pixel inverse-variance weighting of the flux-space term, 2026-09-08.
    # Until now that term used a plain mean over wavelength, i.e. it weighted
    # 3600 A -- where the total throughput is 4.3x worse than at 5000 A -- the
    # same as the middle of the b channel, even though a fixed flux error there
    # is far fewer photons.  The model is the standard-star sensitivity curves
    # in $LVMCORE_DIR/sensitivity: counts = flux/sens, so sigma_flux =
    # sqrt(flux * sens) and w = 1/(flux*sens), with `flux` the TOTAL observed
    # science flux because the whole photon budget sets the noise.  Weights are
    # normalised to mean 1 per ROW, so this changes the weighting ACROSS
    # wavelength without re-weighting rows against each other.
    # See mlp_predictor.noise for the per-arm normalisation, which the
    # published curves do not carry and which is only good to ~1.5x.
    # Set False to A/B against the unweighted term.
    "flux_pixel_weighting": True,
    "flux_pixel_weight_floor_frac": 0.05,   # variance floor, x the row median

    # Moon-down moon amplitude: DERIVED, not learned (2026-09-09).
    # Where the moon-share bracket has collapsed onto amp_prior_floor the
    # decomposition does not measure a moon amplitude at all.  The physics-only
    # model predicts a moon fraction of ~3e-5 in deep dark time, so the
    # bracket sits at the floor (0.02) and the QP sits on that ceiling:
    # measured on gaia-stars-mask with the exact per-row design, 87.5% of
    # moon-down sci rows are pinned, i.e.
    # A_moon = R * A_zodi with R fitted per run.  That rule reproduces the
    # true moon-down amplitude to MAD 0.00051 dex over 7189 rows, against the
    # network's 0.0455 -- the target simply carries no information, and this
    # is 49.7% of the corpus.
    # The shape is not learnable either (93.9% of adjacent Moon_bs knot pairs
    # sit on a beta ratio bound, a corner of the feasible polytope; the SCI
    # and NEAR fits of the SAME exposure disagree at L1 0.311 against a
    # shuffled null of 0.612) but it is only 1.04% of the fitted continuum
    # there, so it is left in the loss rather than dropped.
    #   moon_down_amp_free -> the flux loss is made amplitude-blind on these
    #     rows (prediction rescaled to the true integral, log-amp term
    #     dropped), so only the shape trains.
    #   moon_down_amp_rule -> predict_sci_coefficients_default restores the
    #     amplitude analytically from the PREDICTED zodi.
    # Set both False to A/B against the pre-2026-09-09 behaviour.
    # GATE, revised 2026-09-09 after the first run.  `moon_alt <= 0` is the
    # wrong criterion: scattered moonlight with the moon just below the
    # horizon is real, and the model carries it as
    # horizon_scale = exp(-8.2182 * tanh(depth / 5 deg)) -- only -0.70 dex at
    # -1 deg, -2.72 at -5, saturating at -3.57 near -15.  The rule is valid
    # exactly where the moon-share bracket has collapsed onto amp_prior_floor,
    # i.e. where amp_prior_tol * moon_frac_po <= amp_prior_floor, so the gate
    # tests THAT: moon_frac_po <= 0.02/3.  It beats every altitude cut on both
    # coverage and exactness simultaneously:
    #     moon_alt <= 0            49.7% cover  92.6% exact  7.4% |d|>0.05
    #     moon_alt <= -4           47.0%        96.1%        3.9%
    #     moon_alt <= -12          41.4%        95.7%        4.2%
    #     moon_frac_po <= 0.00667  47.8%        96.2%        3.7%
    # No fixed altitude can match it because the depth term multiplies phase
    # and separation: a thin crescent 2 deg down contributes nothing while a
    # full moon 5 deg down still does.  The gated rows reach up to -0.69 deg
    # while the rows it drops run from -0.0 to -5.32 (median -1.43), and those
    # dropped rows were pinned only 0.4% of the time.
    # moon_down_alt_deg is now only the FALLBACK, used when a triplet has no
    # moon_frac_po (i.e. no v2 model cache).
    # The moon share on gated rows is BIMODAL, not constant: 96.25% sit on the
    # 0.02 ceiling and 3.25% want no moon at all (t = A_moon/(R A_zodi) < 1e-4),
    # with 0.49% in between.  A flat R over-predicts that second mode by the
    # whole component, 2.40% of the sci continuum.  The NEAR ARM knows which
    # mode a row is in -- 96.0% of zero-moon sci rows have the near arm also at
    # t < 0.5, against 0.72% of the pinned ones -- so transfer the measured
    # per-row ratio instead of assuming R:
    #     A_moon(sci) = clip( (A_moon/A_zodi)|near, 0, R ) * A_zodi(sci)
    # No threshold, no mode assignment.  Moon flux error as a fraction of the
    # sci continuum over 6914 gated rows:
    #     flat R * A_zodi      median 0.00086%  p99 2.4038%  >0.5% on 3.47%
    #     near ratio * A_zodi  median 0.00060%  p99 0.0107%  >0.5% on 0.56%
    #     arm-mean ratio       median 0.00059%  p99 0.3959%  >0.5% on 0.81%
    #     copy near verbatim   median 0.10913%  p99 0.9695%  >0.5% on 9.07%
    # i.e. 240 badly-wrong rows become 39, and it beats flat R on the PINNED
    # rows too because the measured ratio absorbs per-row variation.  Near
    # alone beats the arm mean (the far arm's own mode can differ), and
    # copying the near AMPLITUDE is much worse -- it is the RATIO that
    # transfers.  Honest cost: on 22.0% of rows the per-row ratio is noisier
    # than the robust median and slightly worse, median 0.0006% of continuum
    # but p99 1.04% -- those are the ~40 rows where the near arm is zero and
    # the science row is not.  Net absolute error over gated rows falls 87.8%.
    # Set False to A/B against the flat R.
    "moon_down_amp_free": True,
    "moon_down_amp_rule": True,
    "moon_down_ratio_transfer": True,
    "moon_down_frac_max": 0.02 / 3.0,
    "moon_down_alt_deg": 0.0,

    # Bright-moon zodi amplitude: DERIVED, not learned (2026-09-09).  The
    # mirror of the moon-down rule above, on the other half of the corpus.
    # 87.5% of moon-up sci rows sit exactly on the decomposition's Leinert
    # ceiling, so the target there is kappa_z * calibration * Z_pred.  That
    # ceiling is CORRECT: releasing kappa_z on 294 lunation-stratified spectra
    # only re-partitions a conserved continuum (zodi +0.193 dex, moon -0.107,
    # diffuse -0.435, total +0.005) for 0.6% of rms, and the excess tracks the
    # MOON not the ecliptic -- partial rho(excess, FLI | B500) = +0.733 against
    # partial rho(excess, B500 | FLI) = -0.322, the wrong sign for zodiacal
    # light.  So it is moon-into-zodi leakage the bracket exists to stop.
    # Two uses, measured on the FULL corpus (14 457 valid rows):
    #   CLAMP everywhere -- S * Z_pred is a hard upper bound (only 0.01% of
    #     rows exceed it at all, max excess 0.0024 dex), so clipping to it
    #     can only move a prediction toward the truth.
    #   SNAP on gated rows -- fraction of gated rows within 0.01 dex of the
    #     ceiling, with p95/p99 of |log10(S Z / A_zodi)| over them:
    #       moon_alt > 0        50.3% cover  83.9% exact  p95 0.310
    #       moon_frac_po > 0.5  43.8%        94.2%        p95 0.025
    #       moon_frac_po > 0.6  41.5%        97.2%        p95 0.0005
    #       moon_frac_po > 0.7  38.1%        98.6%        p95 0.0002
    #     The every10 subsample is optimistic by ~5 percentiles here and would
    #     have picked 0.5; these are the corpus numbers.
    # The snap is guarded by snap_frac: it fires only where the network
    # already predicts >= 90% of the ceiling, so gated rows that are genuinely
    # interior keep their own answer.  Scored over EVERY valid row with the
    # network simulated at its measured 0.0156 dex scatter:
    #       clamp only            MAD 0.00661  p99 0.0384  max 0.0750
    #       gate 0.6, snap 0.90   MAD 0.00190  p99 0.0375  max 0.0791
    #       no gate,  snap 0.80   MAD 0.00097  p99 0.0813  max 0.1548
    # i.e. 3.5x better MAD and a slightly better p99 for 0.05 dex on the
    # single worst row; dropping the gate trades MAD for a 2x worse p99.
    # zodi_ceiling_amp_free is deliberately False: unlike the moon-down case
    # the bright-moon zodi amplitude is NOT noise, and the network's learned
    # value is exactly what the guard uses to spot interior rows.
    # Needs moon/zodi model cache v2 (v1 stored only the learned-parameter
    # prediction, ~1.5x larger, which hid this effect entirely).
    "zodi_ceiling_rule": True,
    "zodi_ceiling_amp_free": False,
    "zodi_ceiling_gate_frac": 0.6,
    "zodi_ceiling_snap_frac": 0.9,

    "blend_init_alpha": 0.85,
    "alpha_lr_mult": 30.0,          # enough leverage to get the ctx-dependent alpha to learn efficiently
    
    # COEF_ERR weighting (§3.6.3).
    "coef_err_sigma_floor_rel": dict(trainer.DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP),

    # 10-seed ensemble.
    "ensemble_seeds": (42, 43, 44, 45, 46, 47, 48, 49, 50, 51),
    # Train members in parallel worker processes (2026-09-18).  Members are
    # independent given their seed, and the parallel path is verified to give
    # byte-identical members in the requested order.  Worth ~1.5x on a real
    # 300-epoch run: one member already uses about half the GPU, so 2 workers
    # do not halve the time (4 would give ~1.9x).  Set to 1 to train
    # sequentially.  Start method is 'spawn' -- fork dies on a live MPS
    # context; a SCRIPT caller then needs an `if __name__ == "__main__":`
    # guard, a notebook does not.
    "ensemble_workers": 4,
    
    # Isolated zodi branch context restriction (§3.4.1).
    "zodi_ctx_restriction": (
        "airmass", "vanrhijn_285km",
        "ecl_beta_deg", "ecl_lon_sin", "ecl_lon_cos",
        "zodi_log10_v", "sun_sep", "sun_alt", "alt",
        "moon_alt", "moon_sep",
        "moon_phase_sin", "moon_phase_cos",
        "moon_fli", "moon_up_smooth",
        "moon_airmass_up", "moon_signal_proxy",
        # 2026-09-09, and REQUIRED for the zodi ceiling rule to do anything:
        # the zodi head sits on an ISOLATED branch whose context is this
        # whitelist, so a feature missing here never reaches it.  On the 43%
        # of rows where the anchor binds, the target IS
        # S * 10**zodi_po_log10, so this is the target itself rather than a
        # correlate -- it supersedes `zodi_log10_v`, which is a Leinert
        # interpolation with none of the per-row LSF or airmass geometry.
        # `moon_frac_po` is the gate the rule uses, so the head sees the same
        # quantity that decides whether its answer will be snapped.
        "zodi_po_log10", "moon_frac_po",
    ),
    # Phase B (2026-08-26): moon-regime-conditioned isolated head for continuum.
    # Phase A' (2026-08-26): explicit interaction features added below (from
    # residual_ctx_attribution rankings; computed in data._augment_triplet_with_physics_priors).
    "continuum_ctx_restriction": (
        "moon_alt", "moon_sep",
        "moon_phase_sin", "moon_phase_cos",
        "moon_fli", "airmass",
        "moon_fli_x_phase_cos",
        "moon_sig_x_lon_cos", "moon_sig_x_lon_sin",
    ),
    # Phase D (2026-08-26): widened continuum branch (only group with real
    # headroom above sky-arm floor; residual_ctx_attribution RF R^2 = 0.68).
    # Baseline (64, 32) with () extra layers -> 4k params on the branch.
    # Deployed (128, 64) with (64,) extra layer -> 16k params, +12k targeted.
    "continuum_branch_dims": (128, 64),
    "continuum_head_extra_dims": (64,),
    # Context-dependent per-group blend alpha (§3.5).  alpha is the pure
    # AMPLITUDE knob -- it interpolates the two arms' scores and cannot
    # change their colour.  Two things to know before editing this:
    #  * "moon_up_smooth" is IDENTICAL across near/far/sci (it describes the
    #    moon, not the pointing), so it carries no arm information at all.
    #    Median |near-far| as a fraction of each feature's own spread:
    #    moon_sep 0.88, ecl_beta_deg 1.01, airmass 0.65, moon_up_smooth 0.00.
    #  * the predictor reads sci_ctx ONLY.  Feeding it the arm contexts, and
    #    giving it depth, were both measured and both made the moon tail
    #    worse; alpha also already commands 1.3-1.5x the amplitude range the
    #    tail needs, so reach was never the limit.  See ablations.RETIRED.
    "alpha_ctx_features": (
        "moon_up_smooth", "ecl_beta_deg", "airmass",
    ),
    # Restrict ctx-alpha to the anisotropic groups; others keep scalar alpha.
    # Phase F (2026-08-27c): additive moon-zodi coupling.  Keeps both existing
    # paths intact (moon on shared trunk, zodi on isolated branch, continuum on
    # its own branch).  Adds a small coupling branch (64->32) computing a shared
    # latent from the moon-scatter + zodi-geometry ctx union; the latent is
    # projected additively into the moon head output (Linear(32, n_moon)) and
    # the zodi head output (Linear(32, n_zodi)) via zero-initialised linear
    # projectors, so training starts byte-identical to Phase A''.  This is the
    # "preserve zodi isolation while giving moon the shared context" alternative
    # to Phase E (`moon_zodi_mode="shared_branch"`), which was rejected on
    # 2026-08-27b for the atlas-mid regression it caused.  +9k params.
    # "moon_zodi_ctx_restriction": None, # Phase A baseline
    "moon_zodi_ctx_restriction": (
        "airmass", "vanrhijn_285km",
        "ecl_beta_deg", "ecl_lon_sin", "ecl_lon_cos",
        "zodi_log10_v", "sun_sep",
        "moon_alt", "moon_sep",
        "moon_phase_sin", "moon_phase_cos",
        "moon_fli", "moon_up_smooth",
        "moon_airmass_up", "moon_signal_proxy",
        "moon_fli_x_phase_cos",
        "moon_sig_x_lon_cos", "moon_sig_x_lon_sin",
        # 2026-09-09: the physics-only pair, added here too because this
        # branch's whole job is the moon/zodi partition and `moon_frac_po` is
        # exactly the quantity the decomposition's share bracket is stated in.
        # Not speculative: it upgrades `zodi_log10_v` and `moon_signal_proxy`,
        # both already listed, with the same geometry done properly.
        "zodi_po_log10", "moon_frac_po",
    ),
    "moon_zodi_coupling_dims": (64, 32),
    # Off-switch for the coupling above, added 2026-09-04 (the no-coupling
    # path had been deleted, so the old "set moon_zodi_* to None" A/B raised).
    # Ablation A1 kept it ON: disabling costs the moon 4.5pp of its gain over
    # B0_copy_near (+24.0% -> +19.5%) while zodi is flat, so the coupling is
    # the moon-context fix its docstring claims, NOT compensation for the
    # moon/zodi reversals the decomposition used to produce.
    # (Phase E's shared_branch knobs lived here; removed with the other dead
    #  keys below -- the code path was deleted on 2026-08-27.)

    # ------------------------------------------------------------------
    # Removed 2026-09-04: 14 config keys the Trainer does not read.  It
    # warned about them on every run ("NOT read by the trainer and will have
    # no effect"); they were left behind by the 2026-08-27 trim, which deleted
    # the code paths but not the config entries.  Two were active hazards for
    # the post-reversal simplification pass: moon_zodi_mode looked like the
    # coupling off-switch and was not (a real one, moon_zodi_coupling_enabled,
    # had to be added), and alpha_ctx_groups looked like it restricted
    # ctx-alpha but the model hardcodes _ALPHA_CTX_GROUPS = ("moon", "zodi",
    # "continuum") -- same value, so behaviour is unchanged.  An A/B driven
    # through any of these would have measured nothing.  The list lives in
    # mlp_predictor.ablations.DEAD_CONFIG_KEYS; ablations.apply() now refuses
    # any override the trainer does not consume.
    #   alpha_ctx_groups, block_cov_loss_groups, flux_mse_eps_frac,
    #   head_extra_dims, high_airmass_boost, moon_down_ecliptic_beta_deg,
    #   moon_down_ecliptic_boost, moon_zodi_branch_dims, moon_zodi_mode,
    #   moon_zodi_moon_head_extra_dims, moon_zodi_zodi_head_extra_dims,
    #   relative_mse_eps_frac, relative_mse_groups, use_coef_err_weights
    # NOTE use_coef_err_weights was hardcoded True and coef_err_sigma_floor_rel
    # (below) is live, so COEF_ERR weighting remains active.
    # ------------------------------------------------------------------
}
print(f"train_cfg: {len(train_cfg)} knobs, {len(train_cfg['ensemble_seeds'])}-seed ensemble, "
      f"flux_mse_groups={train_cfg['flux_mse_groups']}, "
      f"flux_amp_lambda={train_cfg['flux_amp_lambda']}")


In [ ]:
# --- Fit the ensemble, or restore a previously saved one ---------------------
# Gate: when a saved ensemble exists at SAVED_ENSEMBLE_PATH it is deserialised
# and training is skipped entirely; otherwise the ensemble is trained from
# train_cfg above and the next cell writes it to disk.  Set FORCE_RETRAIN=True
# to retrain even when a saved file is present (e.g. after editing train_cfg).
from pathlib import Path as _Path

from mlp_predictor import serialization

# SAVED_ENSEMBLE_PATH is defined in the imports cell next to DECOMP_DATA_ROOT,
# so the model always lands beside the corpus it was trained on.
# TRUE for this run: flux_amp_lambda changed in the cell above, and that
# changes only the LOSS, not the architecture -- so the saved ensemble would
# deserialise cleanly and you would silently evaluate the OLD weights while
# the new config printed above.  Set back to False once the retrain is saved.
# TRUE for this run: flux_mse_groups changed above.  That changes only the
# LOSS, not the architecture, so the saved ensemble would deserialise
# cleanly and you would silently evaluate the OLD weights against the new
# config.  Set back to False once the retrain is saved.
# TRUE for this run: flux_pixel_weighting was added above (2026-09-08).  Like
# every other flux-loss knob it changes only the LOSS, not the architecture, so
# the saved ensemble would deserialise cleanly and you would silently evaluate
# the OLD weights while the new config printed above.  Set back to False once
# the retrain is saved.
FORCE_RETRAIN = False

ENSEMBLE_WAS_LOADED = (
    _Path(SAVED_ENSEMBLE_PATH).expanduser().exists() and not FORCE_RETRAIN
)

if ENSEMBLE_WAS_LOADED:
    mlp_artifacts = serialization.load_ensemble(SAVED_ENSEMBLE_PATH)
    _ensemble_members = mlp_artifacts["members"]
    # Training-time only; save_ensemble deliberately does not persist it.
    per_seed_test_metrics = None

    # Refuse a model built against a different corpus or ctx layout -- the
    # network's inputs are positional, so a silent mismatch would produce
    # plausible-looking nonsense rather than an error.
    _saved_coef = list(mlp_artifacts["coef_names"])
    _live_coef = [str(n) for n in filtered_triplet["coef_names"]]
    if _saved_coef != _live_coef:
        raise RuntimeError(
            f"Saved ensemble was trained on a different coefficient basis: "
            f"{len(_saved_coef)} names vs {len(_live_coef)} in filtered_triplet. "
            f"Set FORCE_RETRAIN=True or point SAVED_ENSEMBLE_PATH elsewhere.")
    if list(mlp_artifacts["ctx_names"]) != list(filtered_triplet["ctx_names"]):
        raise RuntimeError(
            "Saved ensemble ctx_names differ from filtered_triplet['ctx_names'] "
            "(context augment changed since training). Set FORCE_RETRAIN=True.")

    # The saved payload omits the train/val/test split by design, but the split
    # is exactly reproducible: Trainer.run_ensemble trains every member on
    # filtered_triplet['compress_*_idx'], which cell 8 derives deterministically
    # from (obstime_mjd, moon_phase, seed=42).
    train_idx = np.asarray(filtered_triplet["compress_train_idx"], dtype=int)
    val_idx = np.asarray(filtered_triplet["compress_val_idx"], dtype=int)
    test_idx = np.asarray(filtered_triplet["compress_test_idx"], dtype=int)
    mlp_artifacts["train_idx"] = train_idx
    mlp_artifacts["val_idx"] = val_idx
    mlp_artifacts["test_idx"] = test_idx

    # Prefer the compressor / geometry / grouping the model was TRAINED with over
    # the freshly fitted ones from cell 8: the network's scores are only
    # meaningful under its own compressor.
    group_compressors = mlp_artifacts["compressors"]
    compress_geom_kwargs = mlp_artifacts["geom_kwargs"]
    group_indices = mlp_artifacts["group_indices"]

    print(f"Restored {len(_ensemble_members)}-member ensemble from "
          f"{SAVED_ENSEMBLE_PATH}; training skipped.")
    print(f"  seeds={list(mlp_artifacts['seeds'])}  "
          f"split: train={train_idx.size} val={val_idx.size} test={test_idx.size}")
    print("  using the saved compressors / geom_kwargs / group_indices "
          "(cell 8's fitted copies are overridden).")
    print("  NB per-seed test metrics and training history are not persisted, so "
          "the training-log")
    print("     tables in this cell are unavailable on the load path.")
else:
    raise RuntimeError(
        f"No saved ensemble found at {SAVED_ENSEMBLE_PATH}")

coef_near_all = np.asarray(filtered_triplet["coef_near"], dtype=np.float32)
coef_far_all  = np.asarray(filtered_triplet["coef_far"],  dtype=np.float32)
coef_sci_all  = np.asarray(filtered_triplet["coef_sci"],  dtype=np.float32)
ctx_near_all  = np.asarray(filtered_triplet["ctx_near"],  dtype=np.float32)
ctx_far_all   = np.asarray(filtered_triplet["ctx_far"],   dtype=np.float32)
ctx_sci_all   = np.asarray(filtered_triplet["ctx_sci"],   dtype=np.float32)
coef_err_near_all = np.asarray(
    filtered_triplet.get("coef_err_near", np.full_like(coef_near_all, np.nan)), dtype=np.float32)
coef_err_far_all = np.asarray(
    filtered_triplet.get("coef_err_far", np.full_like(coef_far_all, np.nan)), dtype=np.float32)
coef_err_sci_all = np.asarray(
    filtered_triplet.get("coef_err_sci", np.full_like(coef_sci_all, np.nan)), dtype=np.float32)

coef_pred_det = trainer.predict_sci_coefficients_default(
    mlp_artifacts,
    coef_near_phys=coef_near_all[test_idx],
    coef_far_phys=coef_far_all[test_idx],
    ctx_near_phys=ctx_near_all[test_idx],
    ctx_far_phys=ctx_far_all[test_idx],
    ctx_sci_phys=ctx_sci_all[test_idx],
).astype(np.float32)


In [ ]:
# --- Training progress against epoch (publication layout) -------------------
# Body copied from mlp_predictor/diagnostics_cells/training_history.py and
# re-laid-out for print: train loss and val loss get their own small panels
# instead of sharing one axes, the near-arm blend alpha is a separate figure
# rather than a second subplot column, and neither figure carries a title or
# a per-seed legend (kept: one legend entry per blend-alpha group, since that
# identifies which physical group a line is, not which seed).  Each figure is
# shown via show_with_pdf_export() (imports cell), which adds an Export button
# that writes its CURRENT view -- after any interactive zoom/pan -- to FIGDIR
# as a vector PDF.
# Recomputes nothing: reads the history/blend_history the trainer recorded.
# Text is TeX-rendered: axis titles and legend entries are $...$ strings, which
# Plotly hands to MathJax rather than its own default sans-serif font, and the
# layout font is set to a serif fallback so the plain numeric tick labels
# (never MathJax, Plotly has no LaTeX tick formatter) match that look instead
# of clashing with it.
from collections import defaultdict as _defaultdict

import plotly.graph_objects as _go
from plotly.subplots import make_subplots as _make_subplots

_texfont_th = 'CMU Serif, Latin Modern Roman, Times New Roman, Times, serif'

_members_th = mlp_artifacts.get('members') or []
if not _members_th:
    raise RuntimeError('mlp_artifacts carries no members to plot.')

_hist_th = [list(m.get('history') or []) for m in _members_th]
if not any(_hist_th):
    print('No per-epoch training history on this ensemble: it was saved before '
          '2026-09-23, when save_ensemble began persisting it. Predictions are '
          'unaffected; retrain to record the curves. Skipping the plot.')
else:
    _seeds_th = list(mlp_artifacts.get('seeds') or range(len(_members_th)))
    _best_ep_th = list(mlp_artifacts.get('best_epochs') or
                       [m.get('best_epoch') for m in _members_th])
    _best_vl_th = list(mlp_artifacts.get('best_val_losses') or
                       [m.get('best_val_loss') for m in _members_th])

    # Alpha groups, in the order the trainer snapshots them.  The ctx-dependent
    # groups (moon / zodi / continuum) also carry `<group>_p16` / `_p84`; filter
    # those out of the group list or they plot as separate series.
    _alpha_groups_th = []
    for _bh in (list(m.get('blend_history') or []) for m in _members_th):
        if _bh:
            _keys_th = list(_bh[0].keys())
            _alpha_groups_th = [k for k in _keys_th
                                if k != 'epoch' and not k.endswith(('_p16', '_p84'))]
            break

    # Standard hues first (red/green/blue/black), then their pastel tints for
    # any group beyond the first four.
    _pal_th = ['red', 'green', 'blue', 'black',
              'lightcoral', 'lightgreen', 'lightblue', 'lightgray']


    def _series_th(rows, key):
        return ([r['epoch'] for r in rows if key in r],
                [r[key] for r in rows if key in r])


    # --- Panel: train loss + val loss, each in their own axes, no seed legend
    # All lines drawn first, then every best-val-loss circle drawn last (in its
    # own loop) so no line from another seed can be laid on top of a circle --
    # trace order is z-order in Plotly.
    _fig_loss_th = _make_subplots(rows=1, cols=2, horizontal_spacing=0.12)

    _best_markers_th = []
    for _i_th, (_sd_th, _h_th) in enumerate(zip(_seeds_th, _hist_th)):
        if not _h_th:
            continue
        _x_th, _y_th = _series_th(_h_th, 'train_loss')
        _fig_loss_th.add_trace(_go.Scatter(
            x=_x_th, y=_y_th, mode='lines', showlegend=False,
            line=dict(color='black', width=0.6),
            hovertemplate=(f'seed {_sd_th} train<br>epoch %{{x}}'
                           '<br>train %{y:.6f}<extra></extra>')), row=1, col=1)
        _x_th, _y_th = _series_th(_h_th, 'val_loss')
        _fig_loss_th.add_trace(_go.Scatter(
            x=_x_th, y=_y_th, mode='lines', showlegend=False,
            line=dict(color='black', width=0.6),
            hovertemplate=(f'seed {_sd_th} val<br>epoch %{{x}}'
                           '<br>val %{y:.6f}<extra></extra>')), row=1, col=2)
        # Best epoch: the weights actually kept, which is what the ensemble uses.
        _be_th = _best_ep_th[_i_th] if _i_th < len(_best_ep_th) else None
        _bv_th = _best_vl_th[_i_th] if _i_th < len(_best_vl_th) else None
        if _be_th is not None and _bv_th is not None and _be_th > 0:
            _best_markers_th.append((_sd_th, _be_th, _bv_th))

    for _sd_th, _be_th, _bv_th in _best_markers_th:
        _fig_loss_th.add_trace(_go.Scatter(
            x=[_be_th], y=[_bv_th], mode='markers', showlegend=False,
            marker=dict(color='black', size=9, symbol='circle-open',
                        line=dict(width=2, color='black')),
            hovertemplate=(f'seed {_sd_th} best<br>epoch %{{x}}'
                           '<br>val %{y:.6f}<extra></extra>')), row=1, col=2)

    # automargin: let the axis grow the margin to fit its own title/ticks
    # rather than trust the fixed `b=` guess below, which is what clipped the
    # "epoch" title when the exported size didn't match the on-screen one.
    _fig_loss_th.update_xaxes(title_text=r'$\text{Epoch}$', automargin=True, row=1, col=1)
    _fig_loss_th.update_xaxes(title_text=r'$\text{Epoch}$', automargin=True, row=1, col=2)
    # Log y: the first few epochs are orders of magnitude above the plateau, and
    # on a linear axis they flatten everything that matters into the baseline.
    _fig_loss_th.update_yaxes(title_text=r'$\text{Train loss}$', type='log',
                              automargin=True, row=1, col=1)
    _fig_loss_th.update_yaxes(title_text=r'$\text{Validation loss}$', type='log',
                              automargin=True, row=1, col=2)
    # height/margin.b bumped from 330/60 for the size=16 font: automargin can
    # only redistribute space WITHIN a fixed canvas, not grow it, so a bigger
    # font needs the canvas itself enlarged or its title gets clipped again.
    _fig_loss_th.update_layout(height=380, width=820, showlegend=False,
                               font=dict(family=_texfont_th, size=16),
                               margin=dict(t=20, b=70))
    _fig_loss_th = show_with_pdf_export(_fig_loss_th, FIGDIR / 'training_loss.pdf')

    # --- Panel: near-arm blend alpha, its own figure
    if _alpha_groups_th:
        _fig_alpha_th = _go.Figure()
        # Mean over members: the per-seed spread is small next to the drift, and
        # one line per group per seed would be unreadable at 10 seeds.
        for _gi_th, _g_th in enumerate(_alpha_groups_th):
            _by_ep_th = _defaultdict(list)
            for m in _members_th:
                for _r_th in (m.get('blend_history') or []):
                    if _g_th in _r_th:
                        _by_ep_th[int(_r_th['epoch'])].append(float(_r_th[_g_th]))
            if not _by_ep_th:
                continue
            _eps_th = sorted(_by_ep_th)
            _col_th = _pal_th[_gi_th % len(_pal_th)]
            _fig_alpha_th.add_trace(_go.Scatter(
                x=_eps_th, y=[float(np.mean(_by_ep_th[e])) for e in _eps_th],
                mode='lines', name=fr'$\mathrm{{{_g_th}}}$',
                line=dict(color=_col_th, width=1.6),
                hovertemplate=(f'{_g_th}<br>epoch %{{x}}'
                               '<br>alpha %{y:.3f}<extra></extra>')))
        _fig_alpha_th.update_xaxes(title_text=r'$\text{Epoch}$', automargin=True)
        _fig_alpha_th.update_yaxes(title_text=r'$\alpha$',
                                   automargin=True)
        # height/margin.b bumped from 380/60 for the size=16 font -- see the
        # note by the loss figure's layout call above.
        _fig_alpha_th.update_layout(
            height=420, width=560, margin=dict(t=20, b=70),
            font=dict(family=_texfont_th, size=16),
            # Legend drawn INSIDE the plotting area (paper coords, top-right)
            # instead of Plotly's default slot to the right of the panel.
            legend=dict(x=0.98, y=0.65, xanchor='right', yanchor='top',
                       bgcolor='rgba(255,255,255,0.7)', bordercolor='black',
                       borderwidth=1))
        _fig_alpha_th = show_with_pdf_export(_fig_alpha_th, FIGDIR / 'training_blend_alpha.pdf')

    _rows_th = []
    for _i_th, (_sd_th, _h_th) in enumerate(zip(_seeds_th, _hist_th)):
        if not _h_th:
            continue
        _v_th = [r['val_loss'] for r in _h_th if 'val_loss' in r]
        _be_th = _best_ep_th[_i_th] if _i_th < len(_best_ep_th) else None
        _n_th = len(_h_th)
        _rows_th.append({
            'seed': _sd_th,
            'epochs_run': _n_th,
            'best_epoch': _be_th,
            'best_val': (_best_vl_th[_i_th] if _i_th < len(_best_vl_th) else None),
            'final_val': _v_th[-1] if _v_th else None,
            # How much of the budget ran after the kept weights. Large = patience
            # never fired and the tail was wasted; ~0 = the run was still improving
            # at the end and n_epochs is the binding constraint.
            'epochs_after_best': (None if _be_th is None else _n_th - _be_th),
            'early_stopped': _n_th < int(mlp_artifacts.get('config', {})
                                         .get('n_epochs', _n_th)),
        })
    training_history_result = pd.DataFrame(_rows_th)
    print(f'Per-seed training summary (n_epochs budget = '
          f'{mlp_artifacts.get("config", {}).get("n_epochs", "?")}, '
          f'patience = {mlp_artifacts.get("config", {}).get("patience", "?")}):')
    print(training_history_result.to_string(index=False,
                                            float_format=lambda v: f'{v:.6g}'))

    _still_falling_th = [r['seed'] for r in _rows_th
                         if r['epochs_after_best'] is not None
                         and r['epochs_after_best'] <= 5]
    if _still_falling_th:
        print(f'\n{len(_still_falling_th)}/{len(_rows_th)} seed(s) had their best '
              f'epoch within 5 of the end ({_still_falling_th}): the epoch budget '
              f'is binding, not the patience. Raising n_epochs should still help.')
    else:
        print(f'\nEvery seed found its best epoch with room to spare '
              f'(median {int(np.median([r["epochs_after_best"] for r in _rows_th]))} '
              f'epochs of budget left after it), so n_epochs is not the limit.')


In [ ]:
# --- Build the diagnostics context ---
from mlp_predictor import diagnostics  # local import: survives kernel-restart edge cases

diag_ctx = diagnostics.DiagnosticsContext(
    filtered_triplet=filtered_triplet,
    mlp_artifacts=mlp_artifacts,
    group_compressors=group_compressors,
    group_indices=group_indices,
    geom_kwargs=compress_geom_kwargs,
    ensemble_members=_ensemble_members,
    coef_pred_det=coef_pred_det,
    coef_near_all=coef_near_all,
    coef_far_all=coef_far_all,
    coef_sci_all=coef_sci_all,
    ctx_near_all=ctx_near_all,
    ctx_far_all=ctx_far_all,
    ctx_sci_all=ctx_sci_all,
    coef_err_near_all=coef_err_near_all,
    coef_err_far_all=coef_err_far_all,
    coef_err_sci_all=coef_err_sci_all,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    # Pre-filter triplet + notebook-only bare names the extracted cell bodies read.
    extras={
        "triplet": triplet,
        "coef_wavelengths_a": ext.coef_wavelengths_a,
        "context_cols": list(cfg.data.context_columns),
        "FACTOR": FACTOR,
        "DECOMP_DATA_ROOT": cfg.data.decomp_data_root,
        "DECOMP_STEM": cfg.data.decomp_stem,
        # Moon-model ctx feature: the diagnostic cells rebuild their own
        # triplets, so they must apply the SAME augments or ctx_names will not
        # match the trained ensemble.
        "USE_MOON_MODEL_FEATURE": USE_MOON_MODEL_FEATURE,
        "MOON_MODEL_CORPUS_PREFIX": cfg.data.decomp_prefix,
        "_DECOMP_SUFFIX": cfg.data.decomp_suffix,
        # Per-row telluric transmission for the reconstruction, or None for the
        # production split-zodi variant.  Bound to the EVERY10 stack because
        # that is the file every reconstructing diagnostic indexes.  Built here
        # rather than per cell so the ~100 MB transmission model is loaded once.
        "TELLURIC_ROW_FOR": (
            data.make_telluric_row_lookup(cfg.data.input_fits_for_basis,
                                          decomp_suffix=cfg.data.decomp_suffix)
            if USE_TELLURIC_RECON else None),
        # One representative-row telluric bundle for the things that need a
        # SINGLE basis for the whole corpus -- the integrated-amplitude weights
        # v_g and the flux-space mesospheric companion.  Moon/zodi/diffuse are
        # identical between variants, but OH is not, so these must not be built
        # on the split-zodi basis when the corpus is telluric.
        "TELLURIC_BASIS_KW": (
            data.make_telluric_row_lookup(
                cfg.data.input_fits_for_basis, verbose=False,
                decomp_suffix=cfg.data.decomp_suffix)(
                    'sci', data.telluric_representative_row(
                        cfg.data.input_fits_for_basis))
            if USE_TELLURIC_RECON else None),
        # Spline-knot counts read by full_spectrum_* cell bodies.
        "N_MOON_KNOTS": N_MOON_KNOTS,
        "SPLIT_ZODI": SPLIT_ZODI,
        "N_ZODI_KNOTS": N_ZODI_KNOTS,
        # Per-seed test metrics DataFrame consumed by naive_baseline.
        # None on the load path: per-seed metrics are not persisted.
        "cmp_df": per_seed_test_metrics,
    },
)
diag = diagnostics.Diagnostics(diag_ctx)


## Diagnostics

Ten cells, down from twenty-eight.  `headline_summary` reads the
persisted results of the three cells above it rather than
recomputing, so it cannot disagree with them.  The cells removed on
2026-09-09 were the uncertainty-calibration family (five cells whose
conclusion was always "ensemble spread under-estimates, use
`ensemble_spread_calibration`"), the per-seed and worst-row-stability
checks, three WRMSE-vs-context cells that were never run, and
`residual_ctx_attribution`, whose finding is closed and recorded in
the changelog.  They live on in `mlp_predictor.diagnostics` and can be
called from a scratch cell if a specific question needs them.


In [ ]:
# Targets: ML vs copy_near / near_geo / mean_geo baselines, per group per regime,
#          in TWO spaces: coefficient sRMSE and integrated FLUX amplitude |dlog10|.
# Look for: ML should beat B2_mean_geo on moon+zodi; look at moon_up / moon_down / close_zodi.
# When the two spaces DISAGREE, believe the flux one -- the coefficient sRMSE is an
# unweighted sum over a group, so a 358-coefficient block is owned by whichever few
# coefficients are numerically largest (measured: 93-95% of the ML's mesospheric MSE
# comes from 5 of 358 sticks).  That is why mesospheric reads -12.3% in coefficients
# and +7.5% in flux on gaia-stars-mask-telluric-chi2, and why the group-equal verdict
# is printed in both spaces at the bottom of the cell.
_tmp = diag.naive_baseline()


In [ ]:
# Targets: per-family AMPLITUDE error vs the context that drives that family.
# Amplitude A = sum_lambda f(lambda) = c . B.sum(axis=1), a linear functional of
# the coefficients; error Delta = log10(A_pred/A_true), because the miss is
# multiplicative (83-89% of the tail MSE is removed by a single per-row rescale).
# Four grids: moon vs moon geometry, zodi vs ecliptic geometry, the diffuse
# continuum (HO2/FeO/O2Ac, coloured) and OH (by wavelength band, plus O2_b01)
# vs the airglow and solar-activity terms.
# Look for: a trace departing from zero in some regime = a bias the head has not
# learned.  Flat at zero = that family's brightness is transferred correctly.
_tmp = diag.amplitude_error_vs_ctx()


In [ ]:
# --- Amplitude prediction error vs context (test split) ---------------------
# Companion to the context-vs-amplitude density grid above: same row set (the
# eight physical components), same curated context columns, and now (2026-
# 09-29) the SAME flux-integrated amplitude functional
# A_g = sum_k c_k * v_k, v_k = template_k(lambda).sum() --
# mlp_predictor/diagnostics_cells/amplitude_error_vs_ctx.py's own definition,
# extended here to atomic/ionospheric (that diagnostic only wires up a basis
# for moon/zodi/diffuse/OH). v_k is looked up by COEFFICIENT NAME from the
# decomposer's own static per-family basis matrices, never assumed to share
# row order with group_indices: the decomposer's 'atom'/'orc' family split
# does not match this project's atomic/ionospheric ML grouping (ATOM_N and
# ATOM_Or are 'atom' to the decomposer, 'ionospheric' to the transfer). O2_b01
# has no static template (fit per row as VECTOR_O2) and is dropped from its
# amplitude, so 'mesospheric' here is OH-only flux, labelled accordingly.
#
# This cell needs the trained ensemble -- it shows how well the NETWORK
# predicts the science pointing's amplitude, not the physical decomposition's
# own context dependence (that is the cell above).
#
# Delta_g = log10(A_g^pred / A_g^true), + = over-predicted; SAME sign
# convention as the underlying diagnostic. Test-split rows only
# (filtered_triplet['compress_test_idx']), never seen in training or in the
# compressor fit.

import matplotlib.pyplot as plt
from astropy.io import fits
from sky_decomp.lsf_surface_iterative import SkyDecompLSFSurfaceIterative as _SkyDecompLSF_amperr
from mlp_predictor.data import make_reconstruction_decomposer as _mk_decomp_amperr

required_amperr = ['mlp_artifacts', 'filtered_triplet', 'group_indices',
                   'N_MOON_KNOTS', 'SPLIT_ZODI', 'N_ZODI_KNOTS',
                   'DECOMP_DATA_ROOT', 'DECOMP_STEM']
_missing_amperr = [k for k in required_amperr if k not in globals()
                  or globals()[k] is None]
if _missing_amperr:
    raise RuntimeError('Run the training + wavelengths cells first. Missing: '
                       + ', '.join(_missing_amperr))

N_BINS_AMPERR = 8

# Same amplitude-row definition as the context-vs-amplitude cell.
_AMP_GROUPS_AMPERR = ['moon', 'zodi', 'mesospheric', 'ionospheric', 'atomic']
_DIFFUSE_SPECIES_AMPERR = ['HO2', 'FeO', 'O2Ac']
_ROW_ORDER_AMPERR = (
    [('group', g) for g in ('moon', 'zodi')]
    + [('species', s) for s in _DIFFUSE_SPECIES_AMPERR]
    + [('group', g) for g in _AMP_GROUPS_AMPERR if g not in ('moon', 'zodi')]
)

# Same curated context set as the context-vs-amplitude cell.
_CURATED_CONTEXT_AMPERR = [
    'alt', 'airmass', 'moon_sep', 'moon_alt', 'moon_phase', 'moon_fli',
    'sun_sep', 'sun_alt', 'obstime_day',
    'vanrhijn_87km',
    'ecl_beta_deg', 'ecl_lon',
]
_XRANGE_CLAMP_AMPERR = {'airmass': (1.0, 2.0), 'vanrhijn_87km': (1.0, 2.0)}
_LCO_LON_DEG_AMPERR = -70.68992  # Las Campanas Observatory
_COL_DISPLAY_LABEL_AMPERR = {'obstime_day': 'local hour (0 = midnight)'}
# 'mesospheric' is OH-only now that O2_b01 has dropped out of its flux
# amplitude (see the header comment) -- needs its own display label.
_ROW_DISPLAY_LABEL_AMPERR = {'mesospheric': 'OH', 'mesospheric':'mesos.', 'ionospheric':'ionos.'}

# --- Predict the test split, using the SCIENCE pointing's own decomposition
# as truth (what the transfer is scored against everywhere else too). -------
_te_amperr = np.asarray(filtered_triplet['compress_test_idx'], dtype=int)
_coef_true_amperr = np.asarray(filtered_triplet['coef_sci'], dtype=np.float64)[_te_amperr]
_coef_pred_amperr = np.asarray(trainer.predict_sci_coefficients_default(
    mlp_artifacts,
    coef_near_phys=np.asarray(filtered_triplet['coef_near'], np.float32)[_te_amperr],
    coef_far_phys=np.asarray(filtered_triplet['coef_far'], np.float32)[_te_amperr],
    ctx_near_phys=np.asarray(filtered_triplet['ctx_near'], np.float32)[_te_amperr],
    ctx_far_phys=np.asarray(filtered_triplet['ctx_far'], np.float32)[_te_amperr],
    ctx_sci_phys=np.asarray(filtered_triplet['ctx_sci'], np.float32)[_te_amperr]),
    dtype=np.float64)

_ctx_names_raw_amperr = [str(n) for n in filtered_triplet['ctx_names']]
_ctx_sci_amperr = np.asarray(filtered_triplet['ctx_sci'], dtype=np.float64)[_te_amperr]
_ctx_names_disp_amperr, _ctx_disp_amperr = data._decode_cyclic_context(
    _ctx_names_raw_amperr, _ctx_sci_amperr)

# obstime_day -> hours from local (LCO) midnight; see the context-vs-
# amplitude cell for why (avoids the UTC-day wrap splitting the night).
if 'obstime_day' in _ctx_names_disp_amperr:
    _od_idx_amperr = _ctx_names_disp_amperr.index('obstime_day')
    _utc_deg_amperr = _ctx_disp_amperr[:, _od_idx_amperr].astype(np.float64)
    _local_centered_amperr = (
        (_utc_deg_amperr + _LCO_LON_DEG_AMPERR + 180.0) % 360.0) - 180.0
    _ctx_disp_amperr[:, _od_idx_amperr] = (
        _local_centered_amperr / 15.0).astype(_ctx_disp_amperr.dtype)

_missing_ctx_amperr = [c for c in _CURATED_CONTEXT_AMPERR
                       if c not in _ctx_names_disp_amperr]
if _missing_ctx_amperr:
    print(f'  curated context columns not found (skipped): {_missing_ctx_amperr}')
_cols_amperr = [c for c in _CURATED_CONTEXT_AMPERR if c in _ctx_names_disp_amperr]
_col_idx_amperr = [_ctx_names_disp_amperr.index(c) for c in _cols_amperr]

# --- Flux-integrated amplitude basis -----------------------------------------
# Same construction as the context-vs-amplitude cell: one decomposer instance
# for its static per-family templates, EVERY10 wave grid, generic LSF (or the
# telluric representative-row bundle for a telluric corpus).
with fits.open(f'{DECOMP_DATA_ROOT}/{DECOMP_STEM}_every10.fits') as _h_amperr:
    _w_amperr = np.asarray(_h_amperr['WAVE'].data, dtype=np.float64)
    _wave_basis_amperr = _w_amperr if _w_amperr.ndim == 1 else _w_amperr[0]

_basis_tel_amperr = None
if globals().get('USE_TELLURIC_RECON'):
    _basis_tel_amperr = data.make_telluric_row_lookup(
        cfg.data.input_fits_for_basis, verbose=False,
        decomp_suffix=cfg.data.decomp_suffix)(
            'sci', data.telluric_representative_row(cfg.data.input_fits_for_basis))
    _amp_model_amperr = _mk_decomp_amperr(
        _wave_basis_amperr, n_spline_knots=N_MOON_KNOTS,
        base_dir=data._infer_base_dir_for_reconstruction(),
        split_zodi=SPLIT_ZODI, n_zodi_spline_knots=N_ZODI_KNOTS,
        telluric=_basis_tel_amperr)
else:
    _amp_model_amperr = _SkyDecompLSF_amperr(
        _wave_basis_amperr, lsf_sigma=1.0, n_spline_knots=N_MOON_KNOTS,
        base_dir=data._infer_base_dir_for_reconstruction(),
        split_zodi=SPLIT_ZODI, n_zodi_spline_knots=N_ZODI_KNOTS)

# Per-coefficient-NAME -> v_k = template_k(lambda).sum().
_name_to_v_amperr = {}
for _mat_amperr, _names_amperr in (
    (_amp_model_amperr.matrix_moon, _amp_model_amperr.moon_names),
    (_amp_model_amperr.matrix_zodi, _amp_model_amperr.zodi_names),
    (_amp_model_amperr.matrix_diffuse, _amp_model_amperr.diffuse_names),
    (_amp_model_amperr.matrix_atom, _amp_model_amperr.atom_names),
    (_amp_model_amperr.matrix_orc, _amp_model_amperr.orc_names),
):
    for _row_amperr, _name_amperr in zip(
            np.asarray(_mat_amperr, dtype=np.float64), _names_amperr):
        _name_to_v_amperr[str(_name_amperr).lower()] = float(_row_amperr.sum())
for _i_amperr in range(_amp_model_amperr.matrix_oh.shape[0]):
    _name_to_v_amperr[f'oh_{_i_amperr:03d}'] = float(np.asarray(
        _amp_model_amperr.matrix_oh[_i_amperr], dtype=np.float64).sum())

# --- Per-row signed log amplitude error --------------------------------------
_coef_names_lower_amperr = [str(n).lower() for n in filtered_triplet['coef_names']]
_dropped_warned_amperr = set()


def _find_coef_col_amperr(name):
    lname = name.lower()
    matches = [i for i, n in enumerate(_coef_names_lower_amperr) if n == lname]
    if not matches:
        raise KeyError(f'coefficient {name!r} not in coef_names')
    return matches[0]


def _delta_amperr(values_true, values_pred):
    ok = np.isfinite(values_true) & np.isfinite(values_pred) & \
        (values_true > 0) & (values_pred > 0)
    d = np.full(values_true.shape, np.nan)
    d[ok] = np.log10(values_pred[ok] / values_true[ok])
    return d


def _group_amplitudes_amperr(label):
    """(A_true, A_pred) via the flux-integrated functional, name-aligned."""
    _idx = group_indices.get(label)
    if _idx is None or len(_idx) == 0:
        return None
    _keep, _v = [], []
    _dropped = []
    for _j in _idx:
        _n = _coef_names_lower_amperr[_j]
        if _n in _name_to_v_amperr:
            _keep.append(_j)
            _v.append(_name_to_v_amperr[_n])
        else:
            _dropped.append(_n)
    if _dropped and label not in _dropped_warned_amperr:
        print(f'  {label}: no static template for {_dropped} (per-row-fit '
              'template); excluded from its flux amplitude.')
        _dropped_warned_amperr.add(label)
    _v = np.asarray(_v, dtype=np.float64)
    _at = _coef_true_amperr[:, _keep] @ _v
    _ap = _coef_pred_amperr[:, _keep] @ _v
    return _at, _ap


def _species_amplitudes_amperr(name):
    _ci = _find_coef_col_amperr(name)
    _v = _name_to_v_amperr[name.lower()]
    return _coef_true_amperr[:, _ci] * _v, _coef_pred_amperr[:, _ci] * _v


_delta_rows_amperr = {}
_amp_is_species_amperr = set()
for _kind, _label in _ROW_ORDER_AMPERR:
    if _kind == 'group':
        _res = _group_amplitudes_amperr(_label)
        if _res is None:
            continue
        _at, _ap = _res
    else:
        _at, _ap = _species_amplitudes_amperr(_label)
        _amp_is_species_amperr.add(_label)
    _delta_rows_amperr[_label] = _delta_amperr(_at, _ap)

_row_labels_amperr = list(_delta_rows_amperr)
n_rows_amperr = len(_row_labels_amperr)
n_cols_amperr = len(_cols_amperr)
print(f'Amplitude-error grid: {n_rows_amperr} rows x {n_cols_amperr} cols, '
      f'{_te_amperr.size} test rows (held out from training and from the '
      f'compressor fit).')

# --- Render: median line + 16-84th percentile ("+/-1 sigma") band -----------
_panel_w_in, _panel_h_in = 2.6, 2.2
_fig_w_amperr = _panel_w_in * n_cols_amperr + 1.4
_fig_h_amperr = _panel_h_in * n_rows_amperr + 1.2

fig_amperr, axes_amperr = plt.subplots(
    n_rows_amperr, n_cols_amperr, sharex='col', sharey='row',
    figsize=(_fig_w_amperr, _fig_h_amperr), squeeze=False)

for r_i, _g in enumerate(_row_labels_amperr):
    _d = _delta_rows_amperr[_g]
    for c_i, _cidx in enumerate(_col_idx_amperr):
        ax = axes_amperr[r_i, c_i]
        _col_name_amperr = _cols_amperr[c_i]
        _x = _ctx_disp_amperr[:, _cidx]
        _m = np.isfinite(_x) & np.isfinite(_d)
        _clamp = _XRANGE_CLAMP_AMPERR.get(_col_name_amperr)
        if _clamp is not None:
            _m &= (_x >= _clamp[0]) & (_x <= _clamp[1])
        if _m.sum() >= 40:
            _qs = np.unique(np.nanpercentile(_x[_m], np.linspace(0, 100, N_BINS_AMPERR + 1)))
            _cx, _md, _plo, _phi = [], [], [], []
            for _b in range(len(_qs) - 1):
                _sel = _m & (_x >= _qs[_b]) & (_x <= _qs[_b + 1])
                if _sel.sum() < 8:
                    continue
                _cx.append(float(np.median(_x[_sel])))
                _md.append(float(np.median(_d[_sel])))
                _plo.append(float(np.nanpercentile(_d[_sel], 15.865)))
                _phi.append(float(np.nanpercentile(_d[_sel], 84.135)))
            if len(_cx) > 1:
                ax.fill_between(_cx, _plo, _phi, color='0.6', alpha=0.35,
                                linewidth=0, zorder=1)
                ax.plot(_cx, _md, color='black', linewidth=1.2, marker='o',
                       markersize=2.5, zorder=2)
        ax.axhline(0.0, color='black', linewidth=0.6, linestyle='--', zorder=0)
        ax.tick_params(axis='both', labelsize=9, length=3, pad=2)
        for _sp in ax.spines.values():
            _sp.set_linewidth(0.5)
        _col_label_amperr = _COL_DISPLAY_LABEL_AMPERR.get(
            _col_name_amperr, _col_name_amperr.replace('_', ' '))
        if r_i == 0:
            ax.set_title(_col_label_amperr, fontsize=15)
        if r_i == n_rows_amperr - 1:
            ax.set_xlabel(_col_label_amperr, fontsize=15)
            ax.tick_params(axis='x', labelrotation=45)
        if c_i == 0:
            _g_disp = _ROW_DISPLAY_LABEL_AMPERR.get(_g, _g)
            _ylab = _g_disp if _g in _amp_is_species_amperr else f'A({_g_disp})'
            ax.set_ylabel(f'$\\Delta$ {_ylab} [dex]', fontsize=15)

fig_amperr.subplots_adjust(left=0.045, right=0.995, top=0.93, bottom=0.09,
                           wspace=0.08, hspace=0.18)

fig_amperr.savefig(FIGDIR / 'amplitude_error_vs_context.pdf')
print(f"Saved {FIGDIR / 'amplitude_error_vs_context.pdf'}")

# Inline preview: same horizontal-scroll wrapper as the context-vs-amplitude
# cell (VS Code caps inline image width at the cell width otherwise).
import base64
import io as _io_amperr
from IPython.display import HTML as _HTML_amperr

_buf_amperr = _io_amperr.BytesIO()
fig_amperr.savefig(_buf_amperr, format='png', dpi=110, bbox_inches='tight')
plt.close(fig_amperr)
_buf_amperr.seek(0)
_png_amperr = _buf_amperr.getvalue()
_b64_amperr = base64.b64encode(_png_amperr).decode('ascii')
_display_px_amperr = int(_fig_w_amperr * 70)
display(_HTML_amperr(
    f'<div style="overflow-x: auto; overflow-y: hidden; width: 100%;">'
    f'<img src="data:image/png;base64,{_b64_amperr}" '
    f'style="width: {_display_px_amperr}px; max-width: none; height: auto; '
    f'display: block;" /></div>'))



## Full-spectrum verification


In [ ]:
# Targets: reconstruct one every10 row from pred vs true coefs.
# Look for: quality of the fit; residual panels break out moon / zodi / lines / diffuse.
REQUESTED_ROW = 1368
_tmp = diag.full_spectrum_single_row(row=REQUESTED_ROW, show_moon_zodi_model=True)


In [ ]:
# --- Full pipeline example: near/far decomposition and the predicted,
#     delivered science sky ---------------------------------------------------
# Starting point: mlp_predictor/diagnostics_cells/full_spectrum_single_row.py
# (the cell above calls it) -- this cell rebuilds its reconstructions
# (comps_near_from_near, comps_far_from_far, comps_sci from the ML
# prediction) directly, laid out as:
#   1) near-arm decomposition: observed, its own fit, its own components;
#   2) far-arm decomposition: same, from the far arm's own coefficients;
#   3) science (pred): observed science spectrum against the ML-PREDICTED
#      total reconstruction and its individual predicted components -- this
#      is the actual deliverable, not a comparison against the science
#      field's own (ground-truth) decomposition;
#   4) residual = observed - predicted (the actual sky-subtracted science
#      spectrum);
#   5) three zoom-in panels on the science (pred) panel's own predicted
#      components + observed spectrum, same wavelength windows and styling
#      as the example-decomposition cell's zoom row.
#
# Same colour convention in every panel: observed=black, total
# reconstruction=red, moonlight=blue, zodiacal light=purple, OH + O2
# airglow=light gray, diffuse continuum=orange, atomic + ionospheric
# lines=dark gray.
#
# Two example rows are rendered below (EXAMPLE_SPECS_FULL), side by side for
# comparison while we decide which one to keep in the paper:
#   'typical'           -- the original percentile-of-good-chi2 pick (moon-up)
#   'near_far_contrast' -- a row picked to MAXIMISE the near/far difference
#                          (see the row-selection comment below), moon-down
import numpy as np
from astropy.io import fits
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import decompose_parallel  # science-line mask, as in the example-decomp cell

MOON_UP_ALT_MIN = 20.0         # deg; well up, not a marginal/twilight case
MOON_UP_FLI_MIN = 0.3          # illuminated fraction; visibly bright for the "up" example

# Every10 position 74 (row_index 74), found by an offline search (near/far
# median-flux log-ratio and, from the decomposition coefficients, the
# flux-integrated zodi-amplitude log-ratio -- same functional as cells 6/15 --
# ranked among rows passing a generous per-arm chi2 gate): moon essentially
# off (alt=-32.9 deg, illuminated fraction 0.01), near/far median-flux ratio
# 0.36 dex (~2.3x), near/far zodi-amplitude ratio 0.49 dex (~3.1x) -- the two
# agree in sign and dominate any moon contribution, so this is a clean
# on/off-ecliptic zodi contrast between the two arm pointings, not a moon
# effect. Both arms fit well on their own (chi2 0.0072 near / 0.0039 far),
# though the SCIENCE pointing's own chi2 is worse (0.071) than the 'typical'
# pick, so its ML-predicted residual runs larger -- kept here for comparison
# rather than presumed better.
EXAMPLE_SPECS_FULL = [
    dict(name='typical', moon_state='up', row_mode='typical',
        good_chi2_percentile=25.0, filename='full_pipeline_example.pdf'),
    dict(name='near_far_contrast', moon_state='down', row_mode='fixed_pos',
        fixed_every10_pos=74, filename='full_pipeline_example_near_far_contrast.pdf'),
]

_texfont_full = 'CMU Serif, Latin Modern Roman, Times New Roman, Times, serif'
_COLOR_OBS_FULL = 'black'
_COLOR_TOTAL_FULL = 'red'
_COLOR_MOON_FULL = 'blue'
_COLOR_ZODI_FULL = 'purple'
_COLOR_OH_FULL = 'lightgray'
_COLOR_DIFFUSE_FULL = 'orange'
_COLOR_ATOMIC_FULL = 'dimgray'

required_full = ['mlp_artifacts', 'filtered_triplet', 'N_MOON_KNOTS',
                 'N_ZODI_KNOTS', 'DECOMP_DATA_ROOT', 'DECOMP_STEM']
_missing_full = [k for k in required_full if k not in globals()
                or globals()[k] is None]
if _missing_full:
    raise RuntimeError('Run the training cells first. Missing: '
                       + ', '.join(_missing_full))

_e10_stem_full = f"{DECOMP_DATA_ROOT}/{DECOMP_STEM}_every10"
EVERY10_INPUT_FULL = f"{_e10_stem_full}.fits"
EVERY10_DECOMP_FULL = {
    'near': f"{_e10_stem_full}_decomp_sky1{cfg.data.decomp_suffix}.fits",
    'far':  f"{_e10_stem_full}_decomp_sky2{cfg.data.decomp_suffix}.fits",
    'sci':  f"{_e10_stem_full}_decomp_sci{cfg.data.decomp_suffix}.fits",
}
_FLUX_KEY_FULL = {'near': 'FLUX_SKY_NEAR', 'far': 'FLUX_SKY_FAR', 'sci': 'FLUX_SCI'}

# --- Build the every10 triplet ONCE (mode-independent) ----------------------
# Needs the FULL post-augmentation context (42 columns) -- not just
# moon_alt/moon_phase_cos -- because the ML transfer's own physical-context
# assertion requires it. Same augment calls as cell 1 / full_spectrum_single_row.py,
# applied to the every10 stem instead of the full corpus stem.
e10_triplet_full = data.build_triplet_coef_dataset(
    input_fits_path=EVERY10_INPUT_FULL,
    sky_near_decomp_fits_path=EVERY10_DECOMP_FULL['near'],
    sky_far_decomp_fits_path=EVERY10_DECOMP_FULL['far'],
    sci_decomp_fits_path=EVERY10_DECOMP_FULL['sci'],
    context_columns=list(cfg.data.context_columns),
    return_chi2=True,
)
data._augment_triplet_with_ecliptic(e10_triplet_full, meta_fits_path=EVERY10_INPUT_FULL)
data._augment_triplet_with_physics_priors(e10_triplet_full)
if globals().get('USE_MOON_MODEL_FEATURE', False):
    data._augment_triplet_with_moon_model(e10_triplet_full, _e10_stem_full)
if list(e10_triplet_full['ctx_names']) != list(mlp_artifacts['ctx_names']):
    raise RuntimeError(
        'every10 ctx_names do not match the trained ensemble -- augmentation '
        'flags must match the training-cell settings exactly.')
_ctx_names_full = list(e10_triplet_full['ctx_names'])
_moon_alt_full = np.asarray(
    e10_triplet_full['ctx_sci'][:, _ctx_names_full.index('moon_alt')], dtype=np.float64)
_moon_fli_full = (1.0 - np.asarray(
    e10_triplet_full['ctx_sci'][:, _ctx_names_full.index('moon_phase_cos')],
    dtype=np.float64)) / 2.0
_chi2_full = np.asarray(e10_triplet_full['chi2_sci'], dtype=np.float64)

with fits.open(EVERY10_INPUT_FULL) as _hdul_full:
    _wave_arr_full = np.asarray(_hdul_full['WAVE'].data, dtype=np.float64)
    _lsf_sci_arr_full = np.asarray(_hdul_full['LSF_SCI'].data, dtype=np.float64)
    _flux_all_full = {
        _arm: np.asarray(_hdul_full[_key].data, dtype=np.float64)
        for _arm, _key in _FLUX_KEY_FULL.items()
    }

_base_dir_full = data._infer_base_dir_for_reconstruction()
_tel_lookup_full = (
    data.make_telluric_row_lookup(cfg.data.input_fits_for_basis,
                                  decomp_suffix=cfg.data.decomp_suffix)
    if USE_TELLURIC_RECON else None)
_KIND_MAP_FULL = {'near': 'sky1', 'far': 'sky2', 'sci': 'sci'}

_LOG_FLOOR_FULL = 1e-2  # display units (1e-16 erg/s/cm2/A); see the
                        # example-decomposition cell for why a log axis needs
                        # a floor rather than letting zero/near-zero values
                        # through.


def _floored_full(y):
    return np.clip(np.asarray(y, dtype=np.float64), _LOG_FLOOR_FULL, None)


def _species_dict_full(comps):
    return {
        'moonlight': (comps['moon'], _COLOR_MOON_FULL),
        'zodi': (comps.get('zodi', np.zeros_like(comps['moon'])), _COLOR_ZODI_FULL),
        'oh_o2': (comps['oh'] + comps['o2'], _COLOR_OH_FULL),
        'diffuse': (comps['diffuse'], _COLOR_DIFFUSE_FULL),
        'atomic': (comps['atom'] + comps['orc'], _COLOR_ATOMIC_FULL),
    }


def _mask_to_spans_full(wave, mask):
    spans = []
    start = None
    for i, m in enumerate(mask):
        if m and start is None:
            start = wave[i]
        elif not m and start is not None:
            spans.append((start, wave[i - 1]))
            start = None
    if start is not None:
        spans.append((start, wave[-1]))
    return spans


_SPECIES_LABEL_FULL = {
    'moonlight': r'\text{Moonlight}',
    'zodi': r'\text{Zodiacal light}',
    'oh_o2': r'\text{OH + O}_2\text{ airglow}',
    'diffuse': r'\text{Diffuse continuum}',
    'atomic': r'\text{Atomic + ionospheric lines}',
}
# Draw order (z-order: earlier = further back), same convention as the
# example-decomposition cell: OH and atomic first so the two line forests sit
# under the smoother continuum curves and the observed/total curves.
_DRAW_ORDER_FULL = ['oh_o2', 'atomic', 'moonlight', 'zodi', 'diffuse']

# Zoom windows: same three regions as the example-decomposition cell's own
# zoom row (blue OH band edge / Halpha+[NII] / red OH band), applied here to
# the PREDICTED science components instead of a single channel's own fit.
_ZOOM_RANGES_FULL = [(3900.0, 4200.0), (6200.0, 6500.0), (8400.0, 8700.0)]

_GRID_SPECS_FULL = [
    [{"colspan": 3}, None, None],
    [{"colspan": 3}, None, None],
    [{"colspan": 3}, None, None],
    [{"colspan": 3}, None, None],
    [{}, {}, {}],
]
_GRID_ROW_HEIGHTS_FULL = [1.0, 1.0, 1.0, 0.7, 1.0]


def _select_row_full(spec):
    moon_state = spec['moon_state']
    mask = ((_moon_alt_full > MOON_UP_ALT_MIN) & (_moon_fli_full > MOON_UP_FLI_MIN)
           if moon_state == 'up' else _moon_alt_full <= 0.0)
    idx_candidates = np.flatnonzero(mask & np.isfinite(_chi2_full))
    if idx_candidates.size == 0:
        raise RuntimeError(f'No every10 rows satisfy moon_state={moon_state!r}.')
    if spec['row_mode'] == 'fixed_pos':
        pos = spec['fixed_every10_pos']
        if pos not in idx_candidates:
            raise RuntimeError(
                f'every10 position {pos} no longer satisfies moon_state={moon_state!r} '
                '-- corpus/filters changed since the offline search; re-run it.')
    elif spec['row_mode'] == 'typical':
        target_chi2 = np.percentile(_chi2_full[idx_candidates], spec['good_chi2_percentile'])
        pos = int(idx_candidates[np.argmin(np.abs(_chi2_full[idx_candidates] - target_chi2))])
    else:
        raise ValueError("row_mode must be 'typical' or 'fixed_pos'")
    return pos


def _build_and_export_full_pipeline(spec):
    pos = _select_row_full(spec)
    idx_row = int(e10_triplet_full['row_index'][pos])
    print(f"Row selection [{spec['name']}]: moon_state={spec['moon_state']}, "
          f"every10 row {idx_row}, sci chi2={_chi2_full[pos]:.4g}, "
          f"moon_alt={_moon_alt_full[pos]:.1f} deg, moon_fli={_moon_fli_full[pos]:.2f}")

    wave_row = _wave_arr_full if _wave_arr_full.ndim == 1 else _wave_arr_full[idx_row]
    lsf_row = _lsf_sci_arr_full if _lsf_sci_arr_full.ndim == 1 else _lsf_sci_arr_full[idx_row]
    flux_row = {_arm: _flux_all_full[_arm][idx_row] for _arm in _FLUX_KEY_FULL}

    def _reconstruct_arm(arm, coef_row):
        lsf_state = data.load_lsf_state_if_available(EVERY10_DECOMP_FULL[arm], idx_row)
        o2_vec = data.load_o2_vector_if_available(EVERY10_DECOMP_FULL[arm], idx_row)
        tel = None if _tel_lookup_full is None else _tel_lookup_full(_KIND_MAP_FULL[arm], idx_row)
        return data.reconstruct_with_lsf(
            wave=wave_row, coef=coef_row,
            lsf=lsf_state if lsf_state is not None else (lsf_row / 2.35),
            n_spline_knots=N_MOON_KNOTS, n_zodi_spline_knots=N_ZODI_KNOTS,
            base_dir=_base_dir_full, o2_vector=o2_vec, telluric=tel)

    coef_near_row = np.asarray(e10_triplet_full['coef_near'][pos], dtype=np.float64)
    coef_far_row = np.asarray(e10_triplet_full['coef_far'][pos], dtype=np.float64)

    comps_near = _reconstruct_arm('near', coef_near_row)
    comps_far = _reconstruct_arm('far', coef_far_row)

    # ML transfer: predict the science coefficients from the near/far arms +
    # ctx, reconstruct with the SCIENCE pointing's own LSF/telluric.
    coef_pred_row = np.asarray(trainer.predict_sci_coefficients_default(
        mlp_artifacts,
        coef_near_phys=e10_triplet_full['coef_near'][pos:pos + 1],
        coef_far_phys=e10_triplet_full['coef_far'][pos:pos + 1],
        ctx_near_phys=e10_triplet_full['ctx_near'][pos:pos + 1],
        ctx_far_phys=e10_triplet_full['ctx_far'][pos:pos + 1],
        ctx_sci_phys=e10_triplet_full['ctx_sci'][pos:pos + 1])[0],
        dtype=np.float64)
    comps_sci_pred = _reconstruct_arm('sci', coef_pred_row)

    resid = flux_row['sci'] - np.asarray(comps_sci_pred['total'], dtype=np.float64) / FACTOR

    line_mask, _ = decompose_parallel.science_line_mask(wave_row, lsf_fwhm=lsf_row)
    line_spans = _mask_to_spans_full(wave_row, line_mask)

    fig = make_subplots(rows=5, cols=3, specs=_GRID_SPECS_FULL,
                       row_heights=_GRID_ROW_HEIGHTS_FULL,
                       vertical_spacing=0.045, horizontal_spacing=0.06)

    # Panel 3 uses the ML-PREDICTED science coefficients' own reconstruction --
    # the deliverable -- not the science field's own (ground-truth) decomposition.
    arm_rows = [('near', comps_near, flux_row['near'], 1),
               ('far', comps_far, flux_row['far'], 2),
               ('sci', comps_sci_pred, flux_row['sci'], 3)]

    def _add_flux_panel(species, flux_arr, comps, row, col, showlegend):
        for key in _DRAW_ORDER_FULL:
            y, color = species[key]
            fig.add_trace(go.Scattergl(
                x=wave_row, y=_floored_full(y), mode='lines',
                name=f'${_SPECIES_LABEL_FULL[key]}$',
                line=dict(color=color, width=0.5 if key == 'oh_o2' else 0.9),
                legend='legend' if showlegend else None,
                showlegend=showlegend), row=row, col=col)
        fig.add_trace(go.Scattergl(
            x=wave_row, y=_floored_full(flux_arr * FACTOR), mode='lines',
            name=r'$\text{observed}$', line=dict(color=_COLOR_OBS_FULL, width=0.8),
            legend='legend' if showlegend else None, showlegend=showlegend), row=row, col=col)
        fig.add_trace(go.Scattergl(
            x=wave_row, y=_floored_full(comps['total']), mode='lines',
            name=r'$\text{total reconstruction}$',
            line=dict(color=_COLOR_TOTAL_FULL, width=1.0),
            legend='legend' if showlegend else None, showlegend=showlegend), row=row, col=col)

    for arm, comps, flux_arr, row in arm_rows:
        species = _species_dict_full(comps)
        _add_flux_panel(species, flux_arr, comps, row=row, col=1, showlegend=(row == 1))

    # --- Panel 4: residual = observed - predicted ---------------------------
    # Legend membership (which traces appear, from the loop above) is unrelated
    # to legend PLACEMENT: the legend is a figure-level overlay positioned by
    # paper coordinates in update_layout below, placed here (horizontal, inside
    # this panel's own y-domain) rather than over the flux panels above.
    fig.add_trace(go.Scattergl(
        x=wave_row, y=resid * FACTOR, mode='lines', showlegend=False,
        line=dict(color='black', width=0.6)), row=4, col=1)
    fig.add_hline(y=0, line=dict(color='gray', width=0.6, dash='dash'), row=4, col=1)
    fig.update_yaxes(range=[-1, 6], row=4, col=1)

    # --- Row 5: zoom-ins on the science (pred) panel's own components/observed
    species_sci_pred = _species_dict_full(comps_sci_pred)
    for col, _ in enumerate(_ZOOM_RANGES_FULL, start=1):
        _add_flux_panel(species_sci_pred, flux_row['sci'], comps_sci_pred,
                       row=5, col=col, showlegend=False)

    # --- Science-line mask shading, every panel -----------------------------
    for lo, hi in line_spans:
        for row in range(1, 5):
            fig.add_vrect(x0=lo, x1=hi, row=row, col=1, fillcolor='gray',
                         opacity=0.2, line_width=0, layer='below')
        for col in range(1, 4):
            fig.add_vrect(x0=lo, x1=hi, row=5, col=col, fillcolor='gray',
                         opacity=0.2, line_width=0, layer='below')

    # --- Axes ----------------------------------------------------------------
    # Row names stay short (no inline units): a spelled-out flux unit does not
    # fit, rotated, inside a ~20%-of-figure-height panel without colliding with
    # its neighbours. The unit is instead given once, as a figure-level
    # annotation above panel 1 (see update_layout below).
    row_ylabel = {
        1: r'$\text{Near sky}$', 2: r'$\text{Far sky}$', 3: r'$\text{Predicted sci}$',
        4: r'$\text{Residual}$',
    }
    for row in range(1, 4):
        fig.update_yaxes(title_text=row_ylabel[row], type='log',
                        tickfont=dict(size=16), row=row, col=1)
        fig.update_xaxes(showticklabels=False, tickfont=dict(size=16), row=row, col=1)
    fig.update_yaxes(title_text=row_ylabel[4], tickfont=dict(size=16), row=4, col=1)
    # No x-axis title here: row 4 is no longer the bottommost row (the zoom row
    # is), so its title would land inside the zoom row below it. The tick labels
    # stay (this is the only place the FULL wavelength range is tick-labelled --
    # the zoom columns below are each a narrow slice), and the shared
    # "Wavelength" title moves to the true bottom axis instead.
    fig.update_xaxes(tickfont=dict(size=16), row=4, col=1)

    # Join every full-width panel's x-axis to the first (pan/zoom stays in sync
    # across rows 1-4, same reasoning as the example-decomposition cell's
    # main+residual join). The zoom row (row 5) deliberately keeps its own
    # per-column ranges -- see below -- so it is NOT joined here.
    x1_id = fig.get_subplot(row=1, col=1).xaxis.plotly_name.replace('axis', '')
    for row in range(2, 5):
        fig.update_xaxes(matches=x1_id, row=row, col=1)

    # Zoom panels: each its own wavelength window, but the SAME y-range/scale as
    # the science (pred) panel (row 3, via `matches`) so peak heights are
    # directly comparable -- same convention as the example-decomposition cell's
    # zoom row. No axis titles: tick labels alone make the zoomed range
    # self-evident, and the flux/residual panels above already carry the shared
    # unit label.
    row3_yaxis_id = fig.get_subplot(row=3, col=1).yaxis.plotly_name.replace('axis', '')
    for col, (lo, hi) in enumerate(_ZOOM_RANGES_FULL, start=1):
        fig.update_xaxes(range=[lo, hi], tickfont=dict(size=16), row=5, col=col)
        fig.update_yaxes(type='log', matches=row3_yaxis_id,
                        tickfont=dict(size=16), row=5, col=col)
    # Shared wavelength title on the true bottom axis (the middle zoom column) --
    # it now has the figure's own bottom margin available, unlike row 4 above.
    # `title_font` (not `tickfont`) is what sizes the title text alone, so the
    # tick numbers set uniformly above stay the same size on all three columns.
    fig.update_xaxes(title_text=r'$\text{Wavelength (Å)}$', title_font=dict(size=16),
                    row=5, col=2)

    # Single legend, horizontal, sitting inside the residual panel (row 4) rather
    # than floating over the flux panels above -- there is room there since the
    # only other content in that panel is a thin residual trace and a dashed
    # zero line, and a horizontal layout keeps it from stacking taller than the
    # panel itself.
    y4_top = fig.get_subplot(row=4, col=1).yaxis.domain[1] - 0.005
    fig.update_layout(
        height=1000, width=850, font=dict(family=_texfont_full, size=16),
        margin=dict(t=20, b=70),
        legend=dict(orientation='h', y=y4_top, bgcolor='rgba(255,255,255,0.7)',
                   xanchor='left', x=0.01, yanchor='top', font=dict(size=11)),
    )
    # Flux unit, spelled out once rather than repeated (and overflowing) on every
    # panel's own y-axis title -- same physical unit as the example-decomposition
    # cell's main-panel label. Inside the first panel's top-left corner, which
    # is empty (the OH forest fills the top right), rather than above the figure.
    fig.add_annotation(
        text=r'$\text{flux in units of }10^{-14}\,\mathrm{erg\,s^{-1}\,cm^{-2}\,\text{Å}^{-1}}$',
        xref='x domain', yref='y domain', x=0.01, y=0.98, xanchor='left', yanchor='top',
        showarrow=False, font=dict(size=13), bgcolor='rgba(255,255,255,0.7)')

    return show_with_pdf_export(fig, FIGDIR / spec['filename'])


_full_pipeline_figures = {}
for _spec in EXAMPLE_SPECS_FULL:
    _full_pipeline_figures[_spec['name']] = _build_and_export_full_pipeline(_spec)
fig_full = _full_pipeline_figures[EXAMPLE_SPECS_FULL[0]['name']]

In [ ]:
# Targets: per-row flux-space RMSE (near/far/sci) + absolute photon chi2.
# Look for: sci pRMSE distribution; per-component residual per row.
# size    = rows evaluated.  None (the default) takes EVERY held-out row
#           that passes the gates -- 2900 on the 1.3.2 corpus.  All of them
#           feed every statistic.  An integer caps it, drawn phase-stratified.
# split   = which rows: 'heldout' (validation + test, the default), 'test',
#           'val', or 'all'.  Before 2026-09-23 this cell drew from every
#           gated every10 row -- ~79% training data, and only a tenth of
#           the split even so; 'all' restores the training rows.
# n_workers = processes for the reconstruction loop.  2900 rows take ~5 min
#           on 8 workers against ~30 min serial; the pool falls back to
#           serial on its own if forking fails.  Reads the CORPUS stack,
#           not every10, so it sees the whole split.
# stroked = rows DRAWN as lines; the memory knob, ~0.8 MB each.  Uncapped
#           at size=500 the figure is ~758 MB of JSON and kills the kernel.
# NB do NOT call diag._init_globals() here: it REPLACES the shared
# exec-globals, discarding naive_baseline_result and
# amplitude_error_vs_ctx_result from the cells above, which is exactly what
# headline_summary reads.  Diagnostics.__init__ already initialises them,
# and _run self-heals after an %autoreload.
_tmp = diag.full_spectrum_batch_rmse(stroked=200, n_workers=8, size=None)

In [ ]:
# --- Publication versions of the batch-RMSE diagnostic's two figures -------
# Starting point: mlp_predictor/diagnostics_cells/full_spectrum_batch_rmse.py
# (the cell above calls it, via diag.full_spectrum_batch_rmse) -- rather than
# re-deriving its ~800 lines of reconstruction + plotting logic, this cell
# takes the actual Plotly Figure objects the diagnostic already built.
# `Diagnostics._run` execs the cell body against a persistent globals dict and
# RETURNS THAT DICT WHOLE (mlp_predictor/diagnostics.py), so `_tmp['fig_resid']`
# and `_tmp['_figc']` are the exact objects the diagnostic drew -- restyling
# them cannot drift from what the diagnostic actually computed, unlike a
# from-scratch reimplementation of its reconstruction loop.
#
#   fig_resid (200-object residual panels): every per-row stroked line
#     recoloured to a single gray (was a per-row turbo-colorscale rainbow);
#     the 5th ("lines") panel dropped entirely; everything else -- the +/-1
#     sigma band, the RMS envelope, the right-column median-residual
#     histograms and their sigma markers -- left exactly as the diagnostic
#     drew it, per the request to leave those alone for now.
#   _figc (chi2 panels): the 2x/10x reference lines and the outlier-vs-floor
#     colour split removed from the bottom scatter row (every point one
#     colour, only the 1x/"at the floor" diagonal kept, since a reader is not
#     meant to read individual rows as flagged outliers here) and its marker
#     size reduced so the point cloud does not saturate to a solid blob as
#     quickly; the top histogram row is untouched.
#   Both: subplot titles and the whole-figure title removed (info moves to
#   the LaTeX caption instead, house style for this notebook), and each
#   figure's legend moved from a floating position above the whole figure to
#   sit inside one of its own panels.
import base64
import re

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _decode_plotly_array(v):
    """Undo Plotly's compact {'dtype','bdata'} binary array encoding.

    `go.Figure(other_fig.to_dict())` does not decode these back to plain
    ndarrays on trace-attribute access (unlike the original, un-round-tripped
    figure), so reading .x/.y/.text off a rebuilt trace needs this first.
    """
    if isinstance(v, dict) and 'bdata' in v:
        return np.frombuffer(base64.b64decode(v['bdata']), dtype=v.get('dtype', 'f8'))
    return np.asarray(v) if v is not None else np.asarray([])


def _minmax_decimate(x, y, n_bins):
    """Bin-wise (min, max) decimation: each of `n_bins` equal-width chunks of
    the ORIGINAL point order contributes its two most extreme y-values, in
    their original x-order (so the line does not zig-zag backwards).

    Needed because kaleido's static (SVG, single-process) renderer is far
    more expensive per point than the browser's WebGL Scattergl path this
    figure is built with: 200 stroked rows x 12401 points x 5 panels renders
    fine interactively but made `write_image` spin at ~100% CPU for many
    minutes (measured: it does not even finish inside kaleido's own 90s
    timeout). A plain stride (every Nth point) would risk aliasing away the
    narrow single-pixel residual spikes this figure exists to show; taking
    both the min AND max of each bin keeps those regardless of which side of
    zero they fall on, at 2 points/bin instead of 1.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    n = len(x)
    if n <= n_bins * 2:
        return x, y
    edges = np.linspace(0, n, n_bins + 1).astype(int)
    xs_out = np.empty(n_bins * 2, dtype=x.dtype)
    ys_out = np.empty(n_bins * 2, dtype=y.dtype)
    k = 0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        if hi <= lo:
            continue
        seg_x, seg_y = x[lo:hi], y[lo:hi]
        pos_a, pos_b = int(np.argmin(seg_y)), int(np.argmax(seg_y))
        if pos_a > pos_b:
            pos_a, pos_b = pos_b, pos_a
        xs_out[k], ys_out[k] = seg_x[pos_a], seg_y[pos_a]; k += 1
        xs_out[k], ys_out[k] = seg_x[pos_b], seg_y[pos_b]; k += 1
    return xs_out[:k], ys_out[:k]


# Bins per stroked line. Applied to fig_resid_pub itself (both the
# interactive fig.show() and the exported PDF use the same decimated data) --
# imperceptible at print/screen scale, and it also makes the interactive
# WebGL render lighter, not just the export. 800 bins -> up to 1600
# points/line measured at ~30s for a single write_image of the full 200-line
# x 5-panel figure (~60s with the double-render mitigation below), against a
# multi-minute hang/timeout at the native 12401 points/line.
_DECIMATE_BINS = 800


required_batchfig = ['_tmp']
_missing_batchfig = [k for k in required_batchfig if k not in globals() or globals()[k] is None]
if _missing_batchfig:
    raise RuntimeError('Run the batch-RMSE diagnostic cell above first. Missing: '
                       + ', '.join(_missing_batchfig))

# Deep copies via the dict round-trip: `_tmp` is `diag._globals` itself (the
# SAME persistent dict every diagnostic call reuses), so mutating fig_resid /
# _figc in place would corrupt state a later diagnostic call might still read.
fig_resid_pub = go.Figure(_tmp['fig_resid'].to_dict())
figc_pub = go.Figure(_tmp['_figc'].to_dict())

_GRAY_LINE = 'rgba(120,120,120,0.55)'
# Same font family every other figure in this notebook uses.
_texfont_batch = 'CMU Serif, Latin Modern Roman, Times New Roman, Times, serif'

# ============================================================================
# fig_resid: recolour every stroked per-row line to a single gray. The +/-1
# sigma band and the RMS envelope are go.Scatter (not go.Scattergl), so
# filtering on trace TYPE hits exactly the 200 (n <= stroked) per-row lines
# and nothing else.
# ============================================================================
for _tr in fig_resid_pub.data:
    if _tr.type == 'scattergl':
        _tr.line.color = _GRAY_LINE
        _x_dec, _y_dec = _minmax_decimate(
            _decode_plotly_array(_tr.x), _decode_plotly_array(_tr.y), _DECIMATE_BINS)
        _tr.x, _tr.y = _x_dec, _y_dec

# The empirical 1sigma/2sigma/3sigma percentile markers (vlines in the
# right-column histograms) and the median vline are already distinguished by
# DASH pattern (solid/dash/dot/dashdot) -- the colour coding on top was
# redundant, so flatten it to black. Shapes, not traces (add_vline/add_hline),
# and identified by y1==1 (full-height vlines) to exclude the y0==y1==0
# zero-reference hlines in the col1 spectra panels, which are left alone.
_RED_BAND = 'rgba(214,39,40,0.20)'
for _shp in fig_resid_pub.layout.shapes:
    if _shp.type == 'line' and _shp.y1 == 1:
        _shp.line.color = 'black'
    elif _shp.type == 'rect' and _shp.fillcolor == 'rgba(130,130,130,0.22)':
        # The +/-1 sigma single-fibre-noise band in the histogram panels
        # (a vrect) -- shaded red instead of gray so it reads as a distinct
        # reference from the (now black) percentile lines.
        _shp.fillcolor = _RED_BAND

# Same band, drawn as a filled trace (not a shape) in the col1 spectra
# panels -- same colour, same reasoning.
for _tr in fig_resid_pub.data:
    if _tr.type == 'scatter' and _tr.fill == 'tonexty' and _tr.name == '\u00b11\u03c3 single fibre':
        _tr.fillcolor = _RED_BAND

# Drop every subplot title (the five row titles + the col-2 header) before
# the row-5 rebuild below, so nothing row-5-specific has to be filtered out
# of it separately; the sigma colour-key annotation is the only one worth
# keeping, and it moves inside the big top panel instead of floating above
# the whole figure (see the "big panel" placement further down).
_annos_resid = list(fig_resid_pub.layout.annotations)
_sigma_key_anno = next((a for a in _annos_resid if a.text.strip().startswith('<span')), None)
_sigma_key_text_black = None
if _sigma_key_anno is not None:
    # This annotation's per-span colours (red/orange/gray) matched the vlines
    # above, which are now black too -- flatten every span to black here as
    # well, since dash pattern alone still distinguishes 1/2/3 sigma.
    _sigma_key_text_black = re.sub(r'rgb\(\d+,\s*\d+,\s*\d+\)', 'rgb(0,0,0)',
                                  _sigma_key_anno.text)

# Drop the 5th ("lines") panel entirely. make_subplots numbers axes by
# (row-1)*ncols + col, so rows 1-4 of a 4-row grid get EXACTLY the same axis
# ids (x,x3,x5,x7 / x2,x4,x6,x8) as rows 1-4 of the 5-row grid this figure
# started as -- only row 5's (x9,x10) go away -- so every kept trace/shape
# can be reused as-is with no axis remapping, just a filter.
_ROW5_AXES = {'x9', 'y9', 'x10', 'y10'}
_kept_data = [t for t in fig_resid_pub.data
             if getattr(t, 'xaxis', None) not in _ROW5_AXES
             and getattr(t, 'yaxis', None) not in _ROW5_AXES]
_kept_shapes = [s for s in fig_resid_pub.layout.shapes
               if s.xref.split(' ')[0] not in _ROW5_AXES
               and s.yref.split(' ')[0] not in _ROW5_AXES]

# Fresh 4-row grid for the domains (built with the wider gap from the start,
# rather than the diagnostic's own 0.03 -- tight enough that the histogram
# panels' y-tick labels crowded the spectra panels' right edge -- so there is
# no separate domain-patching step any more). Built via make_subplots and
# kept as that SAME object (not `go.Figure(data=..., layout=...)`, which
# would lose the private grid_ref bookkeeping `update_xaxes(row=, col=)`
# needs below) -- `add_traces` appends the kept traces without touching
# their already-correct xaxis/yaxis, unlike the `.data =` setter, which only
# accepts a permutation of a figure's OWN existing traces.
_HS_NEW = 0.10
fig_resid_pub = make_subplots(rows=4, cols=2, column_widths=[0.82, 0.18],
                              horizontal_spacing=_HS_NEW, vertical_spacing=0.04)
fig_resid_pub.add_traces(_kept_data)
fig_resid_pub.layout.shapes = tuple(_kept_shapes)
# height=1200: the diagnostic's own 1500 (its width was never set explicitly
# either, left at Plotly's default), scaled down for 4 rows instead of 5 --
# a fresh make_subplots grid carries neither, and left unset the whole
# figure collapses to Plotly's ~450 px default, which is why the first
# attempt at this rebuild had every row's y-axis title overlapping the row
# above it.
# showlegend=False, restored: the diagnostic's own figure-level setting
# (which suppressed every trace-level showlegend=True except the ones it
# deliberately re-enabled, namely row 1's copy of the "+/-1 sigma single
# fibre" band) does not survive the fresh make_subplots() above. Left
# unset, that ONE trace's showlegend=True was enough to pop a real,
# separately-positioned Plotly legend back into existence -- exactly the
# floating-legend look this whole cell exists to get rid of. Its swatch is
# folded into the sigma-key annotation below instead.
fig_resid_pub.update_layout(title=None, height=1200, showlegend=False,
                            margin=dict(l=80, r=20, t=20, b=60))

# Pan/zoom stays in sync across the remaining four panels, same as the
# diagnostic's own `matches="x"` -- lost in the rebuild above since a fresh
# make_subplots grid does not know about it.
for _r in range(2, 5):
    fig_resid_pub.update_xaxes(matches='x', row=_r, col=1)

# Axis titles: row 4 is now genuinely the bottom row (it was one row short of
# it, at row 4 of 5, before this panel was dropped), so it carries the
# wavelength / "median residual" titles without any of the shifting a
# 5-row figure needed.
fig_resid_pub.update_yaxes(title_text='Observed - Predicted', row=1, col=1)
fig_resid_pub.update_yaxes(title_text='Moon decomp. - pred.', row=2, col=1)
fig_resid_pub.update_yaxes(title_text='Zodi decomp. - pred.', row=3, col=1)
fig_resid_pub.update_yaxes(title_text='Diffuse decomp. - pred.', row=4, col=1)
fig_resid_pub.update_xaxes(title_text='Wavelength [Å]', row=4, col=1)
fig_resid_pub.update_xaxes(title_text='Median residual', row=4, col=2)

# "+/-1 sigma single fibre" swatch, folded into the SAME annotation as the
# sigma key (both were separate legends before this cell touched them: this
# one a real, default-positioned Plotly legend now suppressed above, the
# other the diagnostic's own top-of-figure annotation) -- one label, one
# spot, instead of two competing for the same corner.
_single_fibre_swatch = f"<span style='color:{_RED_BAND}'>■</span> Single fiber noise"
_legend_key_text = _single_fibre_swatch
if _sigma_key_text_black is not None:
    _legend_key_text += "  &nbsp;&nbsp;  " + _sigma_key_text_black

# Row 1's own axes are the bare 'x'/'y' (make_subplots' first subplot) --
# ample empty space at the top of that panel, unlike the narrow histogram
# column the diagnostic's own sigma-key annotation position pointed at.
fig_resid_pub.add_annotation(
    text=_legend_key_text, showarrow=False,
    font=(_sigma_key_anno.font if _sigma_key_anno is not None else dict(size=11)),
    xref='x domain', yref='y domain',
    x=0.99, y=0.97, xanchor='right', yanchor='top')

fig_resid_pub.update_layout(font=dict(family=_texfont_batch, size=16))
fig_resid_pub = show_with_pdf_export(fig_resid_pub, FIGDIR / 'batch_residual_panels.pdf')

# ============================================================================
# _figc: drop the 2x/10x reference lines and the outlier-vs-floor colour
# split in the bottom (per-row) scatter panels; merge the two point clouds
# into one uniformly-coloured trace, matching each panel's own histogram
# colour (the "at the floor" colour, since that trace already used it).
# ============================================================================
_SCATTER_AXES = ('x3', 'x4')
_points_by_axis = {}
for _tr in figc_pub.data:
    if getattr(_tr, 'xaxis', None) in _SCATTER_AXES and _tr.mode == 'markers':
        _points_by_axis.setdefault(_tr.xaxis, []).append(_tr)


# Smaller than the diagnostic's own marker (4px, reused from the "at the
# floor" trace before) -- at that size the point cloud saturates to a solid
# blob well before the panel's actual row count would suggest, hiding
# density variation across the cloud instead of showing it.
_SCATTER_MARKER_SIZE = 2.5


def _merge_points(_trs):
    _floor_tr = next((t for t in _trs if t.name and 'floor' in t.name), _trs[0])
    _x = np.concatenate([_decode_plotly_array(_t.x) for _t in _trs])
    _y = np.concatenate([_decode_plotly_array(_t.y) for _t in _trs])
    _text = []
    for _t in _trs:
        _text += list(_decode_plotly_array(_t.text)) if _t.text is not None else []
    return go.Scattergl(
        x=_x, y=_y, mode='markers', showlegend=False,
        marker=dict(color=_floor_tr.marker.color, size=_SCATTER_MARKER_SIZE,
                   opacity=_floor_tr.marker.opacity, line=dict(width=0)),
        text=_text, xaxis=_trs[0].xaxis, yaxis=_trs[0].yaxis,
        hovertemplate=_floor_tr.hovertemplate)


_new_data = []
_merged_done = set()
for _tr in figc_pub.data:
    _xid = getattr(_tr, 'xaxis', None)
    if _xid in _SCATTER_AXES and _tr.mode == 'markers':
        if _xid not in _merged_done:
            _new_data.append(_merge_points(_points_by_axis[_xid]))
            _merged_done.add(_xid)
        continue   # drop the rest of this axis's point-cloud traces
    if _xid in _SCATTER_AXES and _tr.mode == 'lines' and _tr.line.dash != 'solid':
        continue   # 2x / 10x reference lines
    _new_data.append(_tr)
# Fresh Figure, not a `.data =` reassignment: Plotly only allows that
# property to be set to a permutation/subset of the SAME trace objects,
# and _new_data introduces brand new merged traces.
figc_pub = go.Figure(data=_new_data, layout=figc_pub.layout)

# Subplot titles (xref='paper') and the outlier-count annotations in the
# scatter panels (xref in {'x3 domain','x4 domain'}, now stale -- they cited
# the 2x/10x lines and the outlier split just removed) are dropped. The
# histogram panels' median/ratio readout (xref in {'x domain','x2 domain'})
# is kept -- it is a statistic, not a title or an outlier marker.
figc_pub.layout.annotations = tuple(
    a for a in figc_pub.layout.annotations if a.xref in ('x domain', 'x2 domain'))
figc_pub.update_layout(title=None, margin=dict(t=20))

# Narrower gap than a first pass at this needed: with the right column's
# y-axis label now suppressed entirely (both rows share one y-axis instead
# of showing the same tick labels/title twice -- see below), there is
# nothing left to crowd into the left panel, so the panels themselves can
# use the space that used to be reserved for it.
# Domains recomputed from scratch (column_widths=[0.5, 0.5], the only other
# number that matters, unchanged).
_HS_NEW_C2 = 0.05
_c2_col1_w = (1.0 - _HS_NEW_C2) * 0.5
_c2_col1_domain = (0.0, _c2_col1_w)
_c2_col2_domain = (_c2_col1_w + _HS_NEW_C2, 1.0)
for _axname in ('xaxis', 'xaxis3'):
    figc_pub.layout[_axname].domain = _c2_col1_domain
for _axname in ('xaxis2', 'xaxis4'):
    figc_pub.layout[_axname].domain = _c2_col2_domain

# Match each ROW's two panels to the same x/y range, so a reader can compare
# full-band and blue-band directly instead of two independently-autoscaled
# axes that only look comparable.
#
# Row 1 (histograms): x-range is already explicit (Tukey-fenced separately
# per column by the diagnostic) -- union it across both columns. y-range
# (row COUNT) was left on autorange, so it has to be worked out from the
# actual bars: same bin edges the diagnostic already gave each trace
# (`xbins`), NOT rebinned, so individual bar WIDTHS still differ a little
# between columns -- only the axis VIEWPORT (and therefore the scale a
# reader judges bar heights against) is shared.
_row1_x_lo = min(figc_pub.layout.xaxis.range[0], figc_pub.layout.xaxis2.range[0])
_row1_x_hi = max(figc_pub.layout.xaxis.range[1], figc_pub.layout.xaxis2.range[1])
_row1_y_max = 0.0
for _tr in figc_pub.data:
    if getattr(_tr, 'xaxis', None) in ('x', 'x2') and _tr.type == 'histogram':
        _xb = _tr.xbins
        _edges = np.linspace(_xb.start, _xb.end, 37)  # 36 bins, matching the diagnostic's own xbins.size = (hi-lo)/36
        _counts, _ = np.histogram(_decode_plotly_array(_tr.x), bins=_edges)
        _row1_y_max = max(_row1_y_max, float(_counts.max()) if _counts.size else 0.0)
for _axname in ('xaxis', 'xaxis2'):
    figc_pub.layout[_axname].range = [_row1_x_lo, _row1_x_hi]
for _axname in ('yaxis', 'yaxis2'):
    figc_pub.layout[_axname].range = [0, _row1_y_max * 1.05]

# Row 2 (scatter, log-log): x- and y-ranges are already explicit and (per
# panel) already equal to each other -- just union across the two columns
# and apply the same [lo, hi] to all four axes.
_row2_lo = min(figc_pub.layout.xaxis3.range[0], figc_pub.layout.xaxis4.range[0])
_row2_hi = max(figc_pub.layout.xaxis3.range[1], figc_pub.layout.xaxis4.range[1])
for _axname in ('xaxis3', 'yaxis3', 'xaxis4', 'yaxis4'):
    figc_pub.layout[_axname].range = [_row2_lo, _row2_hi]

# Actually SHARE the y-axis within each row now, rather than merely giving
# both panels the same range: `matches` ties them (future pan/zoom stays in
# sync too), and with the right column's scale now guaranteed identical to
# the left's, its own tick labels/title are pure repetition -- dropped, so
# the panels get that width back instead of spending it on a second copy of
# the same axis.
figc_pub.layout.yaxis2.matches = 'y'
figc_pub.layout.yaxis2.showticklabels = False
figc_pub.layout.yaxis2.title.text = None
figc_pub.layout.yaxis4.matches = 'y3'
figc_pub.layout.yaxis4.showticklabels = False
figc_pub.layout.yaxis4.title.text = None

# Legend: the only entries left are the two histogram series (decomposition
# self-fit / ML reconstruction) -- move it inside the top-left histogram
# panel instead of floating above the whole figure. Top-left, not top-right,
# since that panel's own stats annotation already sits top-right.
# Legend position needs PAPER coordinates (Plotly legends are a figure-level
# overlay, not attachable to one subplot's domain-relative frame) -- panel
# (1,1) is the first subplot, so its axes are the unnumbered 'xaxis'/'yaxis',
# whose `.domain` survives the to_dict()/Figure() round trip even though
# `get_subplot` (grid_ref-based) does not.
_dom_x1 = figc_pub.layout.xaxis.domain
_dom_y1 = figc_pub.layout.yaxis.domain
figc_pub.update_layout(
    legend=dict(orientation='v', x=_dom_x1[0] + 0.02, xanchor='left',
               y=_dom_y1[1] - 0.02, yanchor='top',
               font=dict(size=10), bgcolor='rgba(255,255,255,0.7)'))

figc_pub.update_layout(font=dict(family=_texfont_batch, size=16))
figc_pub = show_with_pdf_export(figc_pub, FIGDIR / 'batch_chi2_panels.pdf')

## Wavelength systematics and uncertainty


In [ ]:
_tmp = diag.wavelength_residual_atlas()

# --- Publication version of the wavelength residual atlas -------------------
# Same approach as the batch-RMSE cell: take the Figure the diagnostic built
# (`_tmp['_fig']`, from its persistent exec globals) and restyle a copy, so
# nothing drifts from what the diagnostic computed. Only the first two rows
# are kept: the absolute residual for each moon regime (the diagnostic's
# fractional and flux rows are dropped).
#   * titles and subplot titles removed; each panel labelled inside instead;
#   * colours as in the batch figure: dark gray 68% range, lighter gray 95%
#     range, black median, red +/-1 sigma single-fibre noise;
#   * legend inside the top panel;
#   * house font.
import plotly.graph_objects as go
from plotly.subplots import make_subplots

_fig_atlas = go.Figure(_tmp['_fig'].to_dict())

_RED_BAND_ATLAS = 'rgba(214,39,40,0.20)'
_GRAY_BAND_ATLAS = 'rgba(120,120,120,0.35)'
_LIGHT_GRAY_BAND_ATLAS = 'rgba(120,120,120,0.15)'
_REGIME_COLOUR_ATLAS = {'moon down': 'gray', 'moon up': 'black', 'all rows': 'black'}
_texfont_atlas = 'CMU Serif, Latin Modern Roman, Times New Roman, Times, serif'


def _regime_of(name):
    name = (name or '').lower()
    return next((r for r in _REGIME_COLOUR_ATLAS if r in name), None)


# Which regime each panel shows, read off its named traces (the diagnostic
# puts the regime in every trace name), so the labels cannot drift from the
# data if the split changes.
_panel_regime = {}
for _tr in _fig_atlas.data:
    _r = _regime_of(_tr.name)
    if _r is not None and not (_tr.name or '').lower().startswith('truth'):
        _panel_regime[_tr.yaxis] = _r

for _tr in _fig_atlas.data:
    _nm = (_tr.name or '')
    if _tr.line.color and _tr.line.color.startswith('rgba(40,140,60'):
        # +/-1 sigma single-fibre band: edge trace + filled trace.
        if _tr.fill:
            _tr.fillcolor = _RED_BAND_ATLAS
        _tr.name = 'Single fiber noise'
    elif '68% band' in _nm:
        _tr.fillcolor = _GRAY_BAND_ATLAS
        _tr.name = '68% range'
    elif '95% band' in _nm:
        _tr.fillcolor = _LIGHT_GRAY_BAND_ATLAS
        _tr.name = '95% range'
    elif _nm.startswith('median'):
        _tr.line.color = 'black'
        _tr.line.width = 1.2
        _tr.name = 'Median'

# Keep the two absolute-residual rows (y, y2 from the top; moon down, moon up)
# and rebuild them on a fresh 2-row grid. The axis ids of the kept rows are
# the same in both grids, so traces and shapes carry over unchanged.
_KEEP_ATLAS = ('y', 'y2')
_YRANGE_ATLAS = {'y': [-0.3, 0.3], 'y2': [-0.6, 0.6]}
_src_atlas = _fig_atlas
_fig_atlas = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03)
_fig_atlas.add_traces([_tr for _tr in _src_atlas.data if _tr.yaxis in _KEEP_ATLAS])
_fig_atlas.layout.shapes = [_s for _s in _src_atlas.layout.shapes
                            if (_s.yref or '').split(' ')[0] in _KEEP_ATLAS]
_fig_atlas.update_xaxes(range=_src_atlas.layout.xaxis.range)

for _yid in _KEEP_ATLAS:
    _ax = 'yaxis' + _yid[1:]
    _fig_atlas.layout[_ax].title.text = r'$\text{Decomp.} - \text{pred.}\ (10^{-14})$'
    _fig_atlas.layout[_ax].range = _YRANGE_ATLAS[_yid]
    _r = _panel_regime.get(_yid)
    if _r is not None:
        _fig_atlas.add_annotation(
            text=r'$\text{' + ('All observations' if _r == 'all rows' else _r.capitalize()) + '}$',
            xref=f'x{_yid[1:]} domain', yref=f'{_yid} domain', x=0.01, y=0.96,
            xanchor='left', yanchor='top', showarrow=False, font=dict(size=14),
            bgcolor='rgba(255,255,255,0.7)')
_fig_atlas.update_xaxes(title_text=r'$\text{Wavelength (Å)}$', row=2, col=1)

_dom_top = _fig_atlas.layout.yaxis.domain
_fig_atlas.update_layout(
    width=1000, height=700, font=dict(family=_texfont_atlas, size=16),
    margin=dict(t=20, b=70, l=90, r=20), showlegend=True,
    legend=dict(orientation='h', x=0.99, xanchor='right', y=_dom_top[1] - 0.005,
                yanchor='top', font=dict(size=12), bgcolor='rgba(255,255,255,0.7)'))

_fig_atlas = show_with_pdf_export(_fig_atlas, FIGDIR / 'wavelength_residual_atlas.pdf')
